# 01 · Validación, limpieza, EDA y entrega de base para modelado

Este notebook prepara la base de datos inicial del proyecto **BiciMAD Predictor**.

El objetivo es validar las fuentes de datos, descargar los datos necesarios, limpiar la información, integrar variables temporales y meteorológicas, realizar un análisis exploratorio y dejar documentada la base que deberá usar la fase posterior de preprocesamiento y modelado.

### Qué hace este notebook

- Identifica las fuentes de datos utilizadas.
- Descarga las fuentes necesarias en `data/raw`.
- Limpia datasets auxiliares.
- Limpia el histórico principal de estaciones BiciMAD 2022.
- Integra calendario laboral de Madrid.
- Integra meteorología histórica de AEMET.
- Crea una base enriquecida intermedia en `data/interim`.
- Genera una copia de entrada para modelado en `data/processed`.
- Define reglas de limpieza para EDA.
- Analiza riesgos de estaciones casi vacías y casi llenas.
- Deja claro qué base debe usar el siguiente notebook.

### Qué no hace este notebook

- No entrena modelos.
- No define el modelo final.
- No calcula métricas predictivas.
- No realiza división train/test.
- No aplica codificación ni escalado final de variables.
- No crea el target definitivo de Machine Learning.
- No genera un dataset final entrenable; solo entrega una base preparada para que el siguiente notebook construya el dataset de modelado.

La salida recomendada para que continúe la persona encargada de modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`

La versión trazable del pipeline queda conservada en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`


## Índice maestro del notebook

0. Portada y objetivo del notebook  
1. Resumen rápido para el equipo de modelado  
2. Explicación sencilla del problema y del dataset principal  
3. Mapa general del flujo de datos  
4. Inventario de fuentes originales  
5. Qué papel cumple cada dataset  
6. Configuración del proyecto  
7. Seguridad, credenciales y archivos protegidos  
8. Descarga o verificación de datos originales  
9. Validación inicial de datos raw  
10. Limpieza de datasets auxiliares  
    10.1 GBFS estaciones actuales  
    10.2 GBFS estado actual  
    10.3 Histórico diario de viajes  
    10.4 Calendario laboral  
    10.5 Meteorología AEMET  
11. Limpieza del histórico principal BiciMAD 2022  
12. Creación de la base enriquecida intermedia  
13. Validación de la base enriquecida  
14. Contrato de datos para modelado  
15. Reglas de limpieza para EDA  
16. EDA global  
17. EDA por estación  
18. EDA temporal  
19. EDA meteorológica  
20. EDA estación + hora  
21. Hallazgos principales  
22. Limitaciones  
23. Entrega para el siguiente notebook  
24. Archivos generados


## 1. Resumen rápido para el equipo de modelado

El archivo que debe usar la persona encargada de preprocesamiento y modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`

Este archivo contiene registros históricos de estaciones de BiciMAD durante 2022, enriquecidos con calendario laboral de Madrid y meteorología diaria observada de AEMET.

Cada fila representa el estado de una estación concreta de BiciMAD en un momento concreto del tiempo.

> **Cada fila responde a la pregunta:**  
> ¿Cómo estaba una estación concreta de BiciMAD en una fecha y hora concretas?

La versión equivalente y trazable dentro del pipeline de limpieza se guarda también en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

La diferencia entre ambos archivos es de propósito:

- `data/interim/...`: archivo intermedio trazable, útil para auditar la limpieza y el enriquecimiento.
- `data/processed/...`: copia recomendada como punto de entrada para el siguiente notebook de modelado.

Esta base todavía **no es el dataset final de Machine Learning**. No contiene una variable objetivo definitiva, no tiene división train/test y no tiene codificación ni escalado final de variables.

El siguiente notebook deberá partir de esta base para definir el problema predictivo concreto, crear el target, seleccionar variables, dividir los datos temporalmente y entrenar modelos.


### 1.1 Base recomendada para la fase de preprocesamiento y modelado

| Elemento | Valor |
|---|---|
| Archivo recomendado para modelado | `data/processed/station_status_history_2022_modeling_base.csv` |
| Archivo intermedio trazable | `data/interim/bicimad/station_status_history_2022_enriched_base.csv` |
| Granularidad | Estación + momento temporal |
| Identificador principal | `station_id_historical` |
| Fecha/hora principal | `snapshot_datetime` |
| Variables temporales | `date`, `snapshot_hour`, `snapshot_day_of_week`, `snapshot_month` |
| Variables de calendario | `day_type`, `is_working_day`, `is_weekend`, `is_holiday` |
| Variables meteorológicas | temperatura, precipitación, humedad |
| Variables operativas | `num_bikes_available`, `num_docks_available`, `occupation_ratio` |
| Ubicación para uso del equipo de modelado | `data/processed` |
| Responsabilidad del siguiente notebook | crear target, preprocesar variables, dividir datos y entrenar modelos |

Este notebook no decide el target final ni entrena modelos. Su responsabilidad es dejar una base histórica limpia, validada, enriquecida y bien documentada para que el siguiente paso pueda trabajar con seguridad.


## 2. Explicación sencilla del problema y del dataset principal

BiciMAD es un sistema de bicicleta pública. En este tipo de sistema pueden aparecer dos problemas operativos principales:

1. Una estación puede quedarse con muy pocas bicicletas disponibles.
2. Una estación puede quedarse con muy pocos anclajes libres para devolver bicicletas.

Estos problemas no ocurren igual en todas las estaciones ni en todas las horas. Algunas estaciones pueden quedarse casi vacías en determinadas franjas horarias, mientras que otras pueden saturarse y dificultar la devolución de bicicletas.

Por eso, el proyecto necesita una base histórica que permita estudiar patrones por:

- estación,
- fecha,
- hora,
- tipo de día,
- condiciones meteorológicas,
- disponibilidad de bicicletas,
- disponibilidad de anclajes.

La base principal del proyecto se construye a partir del histórico de estado de estaciones de BiciMAD de 2022. A ese histórico se le añaden variables de calendario y meteorología para que el análisis y el futuro modelo tengan más contexto.


## 3. Mapa general del flujo de datos

El camino de los datos en este notebook sigue esta lógica:

~~~text
Datos originales en data/raw
│
├── Histórico de estaciones BiciMAD 2022
│   └── Limpieza
│       └── station_status_history_2022_clean.csv
│
├── Calendario laboral de Madrid
│   └── Limpieza
│       └── madrid_working_calendar_clean.csv
│
├── Clima diario AEMET 2022
│   └── Limpieza y agregación
│       └── aemet_daily_weather_madrid_2022_join_ready.csv
│
└── Unión por fecha
    └── data/interim/bicimad/station_status_history_2022_enriched_base.csv
        └── copia para modelado
            └── data/processed/station_status_history_2022_modeling_base.csv
~~~

No todos los datasets descargados se usan para entrenar un modelo. Algunos sirven para enriquecer el histórico, otros para contexto de negocio y otros para una posible demo futura.

El archivo recomendado para que continúe el equipo de modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`


## 4. Inventario de fuentes originales

Antes de limpiar o transformar datos, es importante saber de dónde viene cada fuente y para qué sirve.

Esta sección crea un inventario de datasets del proyecto. La tabla distingue entre:

- fuentes principales para análisis y modelado,
- fuentes auxiliares para enriquecer datos,
- fuentes útiles para contexto, visualización o demo futura.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from urllib.parse import quote
from io import BytesIO
import json
import os
import re
import sys
import time
import zipfile
import platform
import warnings

import pandas as pd
import numpy as np

try:
    import requests
except ImportError as exc:
    raise ImportError(
        "Falta la librería requests. Instálala con: pip install requests"
    ) from exc

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib no está disponible. El notebook seguirá sin gráficos.")

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


@dataclass
class DataSource:
    """Representa una fuente de datos utilizada en el proyecto."""

    name: str
    source_type: str
    origin: str
    granularity: str
    project_use: str
    raw_file: str
    interim_file: str
    modeling_role: str


class DataSourceRegistry:
    """Inventario de fuentes de datos del proyecto."""

    def __init__(self, sources: list[DataSource]):
        self.sources = sources

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            [
                {
                    "Dataset": source.name,
                    "Tipo": source.source_type,
                    "Origen": source.origin,
                    "Granularidad": source.granularity,
                    "Uso en el proyecto": source.project_use,
                    "Archivo raw": source.raw_file,
                    "Archivo intermedio": source.interim_file,
                    "Papel para modelado": source.modeling_role,
                }
                for source in self.sources
            ]
        )


data_sources = [
    DataSource(
        name="Histórico de estaciones BiciMAD 2022",
        source_type="Fuente principal",
        origin="EMT Madrid / BiciMAD",
        granularity="Estación + momento temporal",
        project_use="Base principal para analizar disponibilidad y saturación por estación.",
        raw_file="data/raw/bicimad/bicimad_station_status_history_2022_raw.zip",
        interim_file="data/interim/bicimad/station_status_history_2022_clean.csv",
        modeling_role="Fuente base para construir el dataset de modelado.",
    ),
    DataSource(
        name="Calendario laboral de Madrid",
        source_type="Fuente auxiliar",
        origin="Portal de datos abiertos del Ayuntamiento de Madrid",
        granularity="Día",
        project_use="Añadir contexto de día laborable, sábado, domingo o festivo.",
        raw_file="data/raw/calendar/madrid_working_calendar_raw.csv",
        interim_file="data/interim/calendar/madrid_working_calendar_clean.csv",
        modeling_role="Enriquece la base principal con variables temporales.",
    ),
    DataSource(
        name="Climatología diaria AEMET 2022",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Día + estación meteorológica",
        project_use="Añadir temperatura, precipitación y humedad observada al histórico 2022.",
        raw_file="data/raw/weather/aemet_daily_climate_selected_stations_2022_raw.json",
        interim_file="data/interim/weather/aemet_daily_weather_madrid_2022_join_ready.csv",
        modeling_role="Enriquece la base principal con variables meteorológicas históricas.",
    ),
    DataSource(
        name="GBFS estaciones actuales BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Estación actual",
        project_use="Referencia actual de estaciones, coordenadas, capacidad y atributos.",
        raw_file="data/raw/bicimad/gbfs_station_information_raw.json",
        interim_file="data/interim/bicimad/stations_clean.csv",
        modeling_role="No es la base de entrenamiento histórica; útil para demo, mapa o referencia actual.",
    ),
    DataSource(
        name="GBFS estado actual BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Foto actual de estación",
        project_use="Consultar disponibilidad actual de bicicletas y anclajes.",
        raw_file="data/raw/bicimad/gbfs_station_status_snapshot_raw.json",
        interim_file="data/interim/bicimad/station_status_snapshot_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; útil para demo o comparación futura.",
    ),
    DataSource(
        name="Histórico diario de viajes BiciMAD",
        source_type="Fuente de contexto",
        origin="EMT Madrid / BiciMAD",
        granularity="Día",
        project_use="Entender la evolución general del uso de BiciMAD.",
        raw_file="data/raw/bicimad/bicimad_daily_trips_raw.csv",
        interim_file="data/interim/bicimad/bicimad_daily_trips_clean.csv",
        modeling_role="Contexto de negocio; no sustituye al histórico por estación.",
    ),
    DataSource(
        name="Inventario de estaciones AEMET",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Estación meteorológica",
        project_use="Seleccionar estaciones meteorológicas cercanas y fiables para Madrid.",
        raw_file="data/raw/weather/aemet_stations_inventory_raw.json",
        interim_file="data/interim/weather/aemet_stations_inventory_clean.csv",
        modeling_role="Apoya la preparación de variables meteorológicas históricas.",
    ),
    DataSource(
        name="Predicción horaria AEMET Madrid",
        source_type="Fuente para demo futura",
        origin="AEMET OpenData",
        granularity="Hora futura",
        project_use="Demostrar cómo una app futura podría consumir meteorología prevista.",
        raw_file="data/raw/weather/aemet_forecast_hourly_madrid_raw.json",
        interim_file="data/interim/weather/aemet_forecast_hourly_madrid_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; sirve para demo o integración futura.",
    ),
]

registry = DataSourceRegistry(data_sources)
data_sources_df = registry.to_dataframe()
display(data_sources_df)


,Dataset,Tipo,Origen,Granularidad,Uso en el proyecto,Archivo raw,Archivo intermedio,Papel para modelado
0,Histórico de estaciones BiciMAD 2022,Fuente principal,EMT Madrid / BiciMAD,Estación + momento temporal,Base principal para analizar disponibilidad y ...,data/raw/bicimad/bicimad_station_status_histor...,data/interim/bicimad/station_status_history_20...,Fuente base para construir el dataset de model...
1,Calendario laboral de Madrid,Fuente auxiliar,Portal de datos abiertos del Ayuntamiento de M...,Día,"Añadir contexto de día laborable, sábado, domi...",data/raw/calendar/madrid_working_calendar_raw.csv,data/interim/calendar/madrid_working_calendar_...,Enriquece la base principal con variables temp...
2,Climatología diaria AEMET 2022,Fuente auxiliar,AEMET OpenData,Día + estación meteorológica,"Añadir temperatura, precipitación y humedad ob...",data/raw/weather/aemet_daily_climate_selected_...,data/interim/weather/aemet_daily_weather_madri...,Enriquece la base principal con variables mete...
3,GBFS estaciones actuales BiciMAD,Fuente de referencia actual,GBFS BiciMAD,Estación actual,"Referencia actual de estaciones, coordenadas, ...",data/raw/bicimad/gbfs_station_information_raw....,data/interim/bicimad/stations_clean.csv,No es la base de entrenamiento histórica; útil...
4,GBFS estado actual BiciMAD,Fuente de referencia actual,GBFS BiciMAD,Foto actual de estación,Consultar disponibilidad actual de bicicletas ...,data/raw/bicimad/gbfs_station_status_snapshot_...,data/interim/bicimad/station_status_snapshot_c...,No se usa para entrenar el histórico 2022; úti...
5,Histórico diario de viajes BiciMAD,Fuente de contexto,EMT Madrid / BiciMAD,Día,Entender la evolución general del uso de BiciMAD.,data/raw/bicimad/bicimad_daily_trips_raw.csv,data/interim/bicimad/bicimad_daily_trips_clean...,Contexto de negocio; no sustituye al histórico...
6,Inventario de estaciones AEMET,Fuente auxiliar,AEMET OpenData,Estación meteorológica,Seleccionar estaciones meteorológicas cercanas...,data/raw/weather/aemet_stations_inventory_raw....,data/interim/weather/aemet_stations_inventory_...,Apoya la preparación de variables meteorológic...
7,Predicción horaria AEMET Madrid,Fuente para demo futura,AEMET OpenData,Hora futura,Demostrar cómo una app futura podría consumir ...,data/raw/weather/aemet_forecast_hourly_madrid_...,data/interim/weather/aemet_forecast_hourly_mad...,No se usa para entrenar el histórico 2022; sir...


## 5. Qué papel cumple cada dataset

El proyecto utiliza varios datasets, pero no todos tienen el mismo papel.

La fuente central es el histórico de estado de estaciones de BiciMAD 2022. Esta fuente indica cuántas bicicletas y anclajes había disponibles en cada estación a lo largo del tiempo.

El calendario laboral y la meteorología diaria no sustituyen a BiciMAD, sino que lo enriquecen. Sirven para añadir contexto: no es lo mismo analizar una estación en un día laborable que en un festivo, ni en un día frío que en un día templado.

Las fuentes GBFS actuales representan una fotografía reciente de la red. Son útiles para una demo, un mapa o una aplicación futura, pero no deben mezclarse automáticamente con el histórico de 2022 sin un trabajo adicional de compatibilidad.

La predicción horaria de AEMET es útil para una posible app futura, porque permite consultar el tiempo previsto. Sin embargo, no se usa para entrenar el histórico de 2022, ya que el entrenamiento necesita variables históricas observadas.

En resumen:

- **Dataset recomendado para el siguiente paso:** `data/processed/station_status_history_2022_modeling_base.csv`.
- **Dataset intermedio trazable:** `data/interim/bicimad/station_status_history_2022_enriched_base.csv`.
- **Datasets auxiliares:** calendario laboral y clima diario observado.
- **Datasets de contexto o demo:** GBFS actual, viajes diarios y predicción horaria.


## 6. Configuración del proyecto

Antes de descargar o transformar datos, necesitamos comprobar que el notebook se está ejecutando dentro del repositorio correcto y que las carpetas principales existen.


In [2]:
@dataclass
class ProjectConfig:
    """Configuración central de rutas del proyecto."""

    project_root: Path

    @property
    def data_dir(self) -> Path:
        return self.project_root / "data"

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / "raw"

    @property
    def interim_dir(self) -> Path:
        return self.data_dir / "interim"

    @property
    def processed_dir(self) -> Path:
        return self.data_dir / "processed"

    @property
    def notebooks_dir(self) -> Path:
        return self.project_root / "notebooks"

    @property
    def src_dir(self) -> Path:
        return self.project_root / "src"

    @property
    def models_dir(self) -> Path:
        return self.project_root / "models"

    @property
    def reports_dir(self) -> Path:
        return self.project_root / "reports"

    @property
    def figures_dir(self) -> Path:
        return self.reports_dir / "figures"

    @property
    def raw_bicimad_dir(self) -> Path:
        return self.raw_dir / "bicimad"

    @property
    def raw_calendar_dir(self) -> Path:
        return self.raw_dir / "calendar"

    @property
    def raw_weather_dir(self) -> Path:
        return self.raw_dir / "weather"

    @property
    def interim_bicimad_dir(self) -> Path:
        return self.interim_dir / "bicimad"

    @property
    def interim_calendar_dir(self) -> Path:
        return self.interim_dir / "calendar"

    @property
    def interim_weather_dir(self) -> Path:
        return self.interim_dir / "weather"

    @property
    def expected_directories(self) -> list[Path]:
        return [
            self.data_dir,
            self.raw_dir,
            self.raw_bicimad_dir,
            self.raw_calendar_dir,
            self.raw_weather_dir,
            self.interim_dir,
            self.interim_bicimad_dir,
            self.interim_calendar_dir,
            self.interim_weather_dir,
            self.processed_dir,
            self.notebooks_dir,
            self.src_dir,
            self.models_dir,
            self.reports_dir,
            self.figures_dir,
        ]


class ProjectRootFinder:
    """Localiza la raíz del repositorio a partir de la carpeta actual."""

    @staticmethod
    def find(start_path: Path | None = None) -> Path:
        current_path = start_path or Path.cwd()

        for path in [current_path, *current_path.parents]:
            if (path / ".git").exists():
                return path

        raise FileNotFoundError(
            "No se ha podido localizar la raíz del repositorio. "
            "Asegúrate de ejecutar este notebook dentro del proyecto clonado."
        )


project_root = ProjectRootFinder.find()
config = ProjectConfig(project_root=project_root)

for directory in config.expected_directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto detectada correctamente:")
print(config.project_root)

print("\nInformación básica del entorno:")
print(f"Python: {sys.version.split()[0]}")
print(f"Sistema operativo: {platform.platform()}")
print(f"Carpeta actual: {Path.cwd()}")


Raíz del proyecto detectada correctamente:
d:\HELEN\PERSONAL\PROYECTO_3.1\bike-sharing-prediction

Información básica del entorno:
Python: 3.14.3
Sistema operativo: Windows-11-10.0.26200-SP0
Carpeta actual: d:\HELEN\PERSONAL\PROYECTO_3.1\bike-sharing-prediction\notebooks


## 7. Seguridad, credenciales y archivos protegidos

El proyecto usa la API de AEMET para descargar datos meteorológicos.

La clave real de AEMET debe guardarse en un archivo local llamado `.env`.

Ese archivo no debe subirse a GitHub porque contiene información privada. El repositorio incluye `.env.example` como plantilla para que cada persona cree su propio `.env`.

Este notebook nunca imprime la clave real. Solo comprueba si existe y si parece estar configurada.

### 7.1 Cómo obtener y configurar la API Key de AEMET

Página oficial de AEMET OpenData:  
https://opendata.aemet.es/

Página oficial para obtener la API Key:  
https://opendata.aemet.es/centrodedescargas/obtencionAPIKey

### 7.2 Crear el archivo `.env` de forma segura

Ejecuta en terminal desde la raíz del repositorio:

~~~bash
read -r -s -p "Introduce tu AEMET_API_KEY: " AEMET_API_KEY
echo
umask 077
printf "AEMET_API_KEY=%s\n" "$AEMET_API_KEY" > .env
unset AEMET_API_KEY
chmod 600 .env
echo ".env creado con permisos seguros."
ls -l .env
~~~

### 7.3 Confirmar que `.env` no aparece en Git

~~~bash
git status -sb
~~~

Si `.gitignore` está bien configurado, `.env` no debe aparecer como archivo pendiente.


In [3]:
class EnvironmentManager:
    """Gestiona comprobaciones básicas de archivos de entorno sin mostrar secretos."""

    def __init__(self, config: ProjectConfig):
        self.config = config
        self.env_path = self.config.project_root / ".env"
        self.env_example_path = self.config.project_root / ".env.example"

    def env_file_exists(self) -> bool:
        return self.env_path.exists()

    def env_example_exists(self) -> bool:
        return self.env_example_path.exists()

    def read_env_variable(self, variable_name: str) -> str | None:
        if not self.env_file_exists():
            return None

        for line in self.env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)

            if key.strip() == variable_name:
                return value.strip()

        return None

    def check_aemet_api_key(self) -> dict:
        api_key = self.read_env_variable("AEMET_API_KEY")

        return {
            ".env existe": self.env_file_exists(),
            ".env.example existe": self.env_example_exists(),
            "AEMET_API_KEY configurada": bool(api_key),
            "Longitud de AEMET_API_KEY": len(api_key) if api_key else 0,
        }


class GitIgnoreChecker:
    """Comprueba que .gitignore protege los elementos críticos del proyecto."""

    def __init__(self, config: ProjectConfig):
        self.gitignore_path = config.project_root / ".gitignore"

    def check_patterns(self, patterns: list[str]) -> pd.DataFrame:
        content = self.gitignore_path.read_text(encoding="utf-8") if self.gitignore_path.exists() else ""

        return pd.DataFrame(
            [
                {
                    "Patrón": pattern,
                    "Encontrado en .gitignore": pattern in content,
                }
                for pattern in patterns
            ]
        )


environment_manager = EnvironmentManager(config)
environment_status = environment_manager.check_aemet_api_key()

print("Estado de configuración de AEMET:")
display(pd.DataFrame([environment_status]))

if environment_status["AEMET_API_KEY configurada"]:
    print("La clave de AEMET está configurada localmente. No se muestra por seguridad.")
else:
    print("La clave de AEMET no está configurada todavía en .env. Las descargas de AEMET quedarán pendientes.")

protected_patterns = [
    ".env",
    "data/raw/**",
    "data/interim/**",
    "data/processed/**",
    "models/**",
    ".ipynb_checkpoints/",
]

gitignore_checker = GitIgnoreChecker(config)
display(gitignore_checker.check_patterns(protected_patterns))


Estado de configuración de AEMET:


,.env existe,.env.example existe,AEMET_API_KEY configurada,Longitud de AEMET_API_KEY
0,True,True,True,292


La clave de AEMET está configurada localmente. No se muestra por seguridad.


,Patrón,Encontrado en .gitignore
0,.env,True
1,data/raw/**,True
2,data/interim/**,True
3,data/processed/**,True
4,models/**,True
5,.ipynb_checkpoints/,True


## 8. Descarga o verificación de datos originales

En esta sección se descargan o verifican las fuentes originales que se usarán en el proyecto.

Los datos originales se guardan en `data/raw` y no se modifican directamente. Algunas fuentes pueden pesar bastante, especialmente el histórico de BiciMAD 2022. Esto es normal en un proyecto real de datos.

Los datos descargados no se suben a GitHub porque están protegidos por `.gitignore`.

### 8.1 Parámetros de ejecución


In [4]:
RUN_DOWNLOADS = True
RUN_HEAVY_DOWNLOADS = True
FORCE_REDOWNLOAD = False

HTTP_TIMEOUT_SECONDS = 180
AEMET_WAIT_SECONDS = 1.0

print("Parámetros de ejecución:")
print(f"RUN_DOWNLOADS: {RUN_DOWNLOADS}")
print(f"RUN_HEAVY_DOWNLOADS: {RUN_HEAVY_DOWNLOADS}")
print(f"FORCE_REDOWNLOAD: {FORCE_REDOWNLOAD}")
print("\nAviso: algunas fuentes pueden pesar bastante. Los archivos se guardarán localmente en data/raw.")


Parámetros de ejecución:
RUN_DOWNLOADS: True
RUN_HEAVY_DOWNLOADS: True
FORCE_REDOWNLOAD: False

Aviso: algunas fuentes pueden pesar bastante. Los archivos se guardarán localmente en data/raw.


### 8.2 Catálogo de fuentes raw esperadas


In [5]:
@dataclass(frozen=True)
class RawDatasetSpec:
    dataset: str
    source_owner: str
    source_url: str
    local_path: Path
    requires_api_key: bool
    is_heavy_download: bool
    required_for_modeling_base: bool
    access_note: str


class RawDatasetCatalog:
    def __init__(self, specs: list[RawDatasetSpec]):
        self.specs = specs

    def to_dataframe(self) -> pd.DataFrame:
        records = []
        for spec in self.specs:
            exists = spec.local_path.exists()
            size_mb = round(spec.local_path.stat().st_size / 1024 / 1024, 2) if exists else 0
            records.append(
                {
                    "Dataset": spec.dataset,
                    "Fuente": spec.source_owner,
                    "URL o endpoint de referencia": spec.source_url,
                    "Ruta raw esperada": str(spec.local_path.relative_to(config.project_root)),
                    "Existe localmente": exists,
                    "Tamaño MB": size_mb,
                    "Requiere API Key": spec.requires_api_key,
                    "Descarga pesada": spec.is_heavy_download,
                    "Necesario para base de modelado": spec.required_for_modeling_base,
                    "Nota de acceso": spec.access_note,
                }
            )
        return pd.DataFrame(records)


raw_dataset_specs = [
    RawDatasetSpec(
        dataset="GBFS estaciones actuales BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://madrid.publicbikesystem.net/customer/gbfs/v2/es/station_information",
        local_path=config.raw_bicimad_dir / "gbfs_station_information_raw.json",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Feed GBFS actual. Útil para referencia, mapa o demo.",
    ),
    RawDatasetSpec(
        dataset="GBFS estado actual BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://madrid.publicbikesystem.net/customer/gbfs/v2/es/station_status",
        local_path=config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Feed GBFS actual. Fotografía de disponibilidad.",
    ),
    RawDatasetSpec(
        dataset="Histórico diario de viajes BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://datos.emtmadrid.es/dataset/a3442c30-ee33-434f-998d-c63a51a8c446/resource/883ba442-76f4-4969-b5b2-1e9d50df2f86/download/historicoviajesdia.csv",
        local_path=config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Serie diaria de viajes. Sirve como contexto de negocio.",
    ),
    RawDatasetSpec(
        dataset="Histórico de estaciones BiciMAD 2022",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://media.emtmadrid.es/-uHaW6iZkhG",
        local_path=config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip",
        requires_api_key=False,
        is_heavy_download=True,
        required_for_modeling_base=True,
        access_note="Descarga pesada. Se usará la parte de estado de estaciones 2022.",
    ),
    RawDatasetSpec(
        dataset="Calendario laboral de Madrid",
        source_owner="Ayuntamiento de Madrid",
        source_url="https://datos.madrid.es/dataset/300082-0-calendario_laboral/resource/300082-1-calendario_laboral-csv/download/300082-1-calendario_laboral-csv.csv",
        local_path=config.raw_calendar_dir / "madrid_working_calendar_raw.csv",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=True,
        access_note="Calendario laboral 2013-2026.",
    ),
    RawDatasetSpec(
        dataset="Inventario de estaciones AEMET",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones",
        local_path=config.raw_weather_dir / "aemet_stations_inventory_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Permite seleccionar estaciones meteorológicas cercanas y fiables.",
    ),
    RawDatasetSpec(
        dataset="Climatología diaria AEMET 2022",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{indicativo}",
        local_path=config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=True,
        access_note="Datos meteorológicos diarios observados de 2022 para estaciones seleccionadas.",
    ),
    RawDatasetSpec(
        dataset="Predicción horaria AEMET Madrid",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/prediccion/especifica/municipio/horaria/28079",
        local_path=config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Predicción horaria para demo futura.",
    ),
]

raw_catalog = RawDatasetCatalog(raw_dataset_specs)
display(raw_catalog.to_dataframe())


,Dataset,Fuente,URL o endpoint de referencia,Ruta raw esperada,Existe localmente,Tamaño MB,Requiere API Key,Descarga pesada,Necesario para base de modelado,Nota de acceso
0,GBFS estaciones actuales BiciMAD,EMT Madrid / BiciMAD,https://madrid.publicbikesystem.net/customer/g...,data\raw\bicimad\gbfs_station_information_raw....,True,0.51,False,False,False,"Feed GBFS actual. Útil para referencia, mapa o..."
1,GBFS estado actual BiciMAD,EMT Madrid / BiciMAD,https://madrid.publicbikesystem.net/customer/g...,data\raw\bicimad\gbfs_station_status_snapshot_...,True,0.27,False,False,False,Feed GBFS actual. Fotografía de disponibilidad.
2,Histórico diario de viajes BiciMAD,EMT Madrid / BiciMAD,https://datos.emtmadrid.es/dataset/a3442c30-ee...,data\raw\bicimad\bicimad_daily_trips_raw.csv,True,0.01,False,False,False,Serie diaria de viajes. Sirve como contexto de...
3,Histórico de estaciones BiciMAD 2022,EMT Madrid / BiciMAD,https://media.emtmadrid.es/-uHaW6iZkhG,data\raw\bicimad\bicimad_station_status_histor...,True,300.74,False,True,True,Descarga pesada. Se usará la parte de estado d...
4,Calendario laboral de Madrid,Ayuntamiento de Madrid,https://datos.madrid.es/dataset/300082-0-calen...,data\raw\calendar\madrid_working_calendar_raw.csv,True,0.16,False,False,True,Calendario laboral 2013-2026.
5,Inventario de estaciones AEMET,AEMET OpenData,https://opendata.aemet.es/opendata/api/valores...,data\raw\weather\aemet_stations_inventory_raw....,True,0.18,True,False,False,Permite seleccionar estaciones meteorológicas ...
6,Climatología diaria AEMET 2022,AEMET OpenData,https://opendata.aemet.es/opendata/api/valores...,data\raw\weather\aemet_daily_climate_selected_...,True,1.17,True,False,True,Datos meteorológicos diarios observados de 202...
7,Predicción horaria AEMET Madrid,AEMET OpenData,https://opendata.aemet.es/opendata/api/predicc...,data\raw\weather\aemet_forecast_hourly_madrid_...,True,0.04,True,False,False,Predicción horaria para demo futura.


### 8.3 Funciones de descarga


In [6]:
class LocalFileDownloader:
    """Descarga archivos públicos a rutas locales."""

    def __init__(self, timeout_seconds: int = 180):
        self.timeout_seconds = timeout_seconds

    def download(self, url: str, output_path: Path, force: bool = False) -> bool:
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if output_path.exists() and not force:
            print(f"Ya existe: {output_path.relative_to(config.project_root)}")
            return False

        print(f"Descargando: {url}")
        print(f"Destino: {output_path.relative_to(config.project_root)}")

        with requests.get(url, stream=True, timeout=self.timeout_seconds) as response:
            response.raise_for_status()
            with output_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)

        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f"Descarga completada: {size_mb:.2f} MB")
        return True


class AEMETOpenDataClient:
    """Cliente mínimo para descargar datos desde AEMET OpenData."""

    def __init__(self, api_key: str, timeout_seconds: int = 180, wait_seconds: float = 1.0):
        if not api_key:
            raise ValueError("No se ha encontrado AEMET_API_KEY en .env.")
        self.api_key = api_key
        self.timeout_seconds = timeout_seconds
        self.wait_seconds = wait_seconds

    def request_metadata(self, endpoint: str) -> dict:
        response = requests.get(
            endpoint,
            params={"api_key": self.api_key},
            timeout=self.timeout_seconds,
        )
        response.raise_for_status()
        return response.json()

    def download_endpoint_json(self, endpoint: str):
        metadata = self.request_metadata(endpoint)
        if "datos" not in metadata:
            raise ValueError(f"La respuesta de AEMET no contiene clave 'datos': {metadata}")
        data_url = metadata["datos"]
        data_response = requests.get(data_url, timeout=self.timeout_seconds)
        data_response.raise_for_status()
        time.sleep(self.wait_seconds)
        return data_response.json()

    def download_to_file(self, endpoint: str, output_path: Path, force: bool = False) -> bool:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if output_path.exists() and not force:
            print(f"Ya existe: {output_path.relative_to(config.project_root)}")
            return False
        print(f"Descargando AEMET: {endpoint}")
        data = self.download_endpoint_json(endpoint)
        output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f"Descarga AEMET completada: {output_path.relative_to(config.project_root)} ({size_mb:.2f} MB)")
        return True


def get_aemet_api_key(environment_manager: EnvironmentManager) -> str | None:
    return environment_manager.read_env_variable("AEMET_API_KEY")


### 8.4 Descarga de fuentes públicas principales


In [7]:
public_downloader = LocalFileDownloader(timeout_seconds=HTTP_TIMEOUT_SECONDS)

if not RUN_DOWNLOADS:
    print("RUN_DOWNLOADS está desactivado. No se descargan fuentes públicas.")
else:
    public_sources = [spec for spec in raw_dataset_specs if not spec.requires_api_key]

    for spec in public_sources:
        if spec.is_heavy_download and not RUN_HEAVY_DOWNLOADS:
            print(f"Saltando descarga pesada: {spec.dataset}")
            continue

        try:
            public_downloader.download(
                url=spec.source_url,
                output_path=spec.local_path,
                force=FORCE_REDOWNLOAD,
            )
        except Exception as error:
            print(f"No se pudo descargar {spec.dataset}: {error}")


Ya existe: data\raw\bicimad\gbfs_station_information_raw.json
Ya existe: data\raw\bicimad\gbfs_station_status_snapshot_raw.json
Ya existe: data\raw\bicimad\bicimad_daily_trips_raw.csv
Ya existe: data\raw\bicimad\bicimad_station_status_history_2022_raw.zip
Ya existe: data\raw\calendar\madrid_working_calendar_raw.csv


### 8.5 Descarga de fuentes AEMET


In [8]:
selected_aemet_station_ids = ["3126Y", "3129", "3195", "3196", "3200"]

# Se usan rangos mensuales para evitar problemas de límite de rango en AEMET
# y para hacer la descarga más recuperable si una petición falla.
aemet_daily_ranges_2022 = [
    ("2022-01-01T00:00:00UTC", "2022-01-31T23:59:59UTC"),
    ("2022-02-01T00:00:00UTC", "2022-02-28T23:59:59UTC"),
    ("2022-03-01T00:00:00UTC", "2022-03-31T23:59:59UTC"),
    ("2022-04-01T00:00:00UTC", "2022-04-30T23:59:59UTC"),
    ("2022-05-01T00:00:00UTC", "2022-05-31T23:59:59UTC"),
    ("2022-06-01T00:00:00UTC", "2022-06-30T23:59:59UTC"),
    ("2022-07-01T00:00:00UTC", "2022-07-31T23:59:59UTC"),
    ("2022-08-01T00:00:00UTC", "2022-08-31T23:59:59UTC"),
    ("2022-09-01T00:00:00UTC", "2022-09-30T23:59:59UTC"),
    ("2022-10-01T00:00:00UTC", "2022-10-31T23:59:59UTC"),
    ("2022-11-01T00:00:00UTC", "2022-11-30T23:59:59UTC"),
    ("2022-12-01T00:00:00UTC", "2022-12-31T23:59:59UTC"),
]


def json_file_has_records(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False

    if isinstance(data, list):
        return len(data) > 0

    if isinstance(data, dict):
        return len(data) > 0

    return False


def download_aemet_endpoint_json_with_retries(
    client: AEMETOpenDataClient,
    endpoint: str,
    attempts: int = 3,
    wait_seconds: float = 3.0,
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            return client.download_endpoint_json(endpoint)
        except Exception as error:
            last_error = error
            print(f"Intento {attempt}/{attempts} fallido: {error}")

            if attempt < attempts:
                time.sleep(wait_seconds)

    raise last_error


if not RUN_DOWNLOADS:
    print("RUN_DOWNLOADS está desactivado. No se descargan fuentes AEMET.")
else:
    aemet_api_key = get_aemet_api_key(environment_manager)

    if not aemet_api_key:
        print("No se ha encontrado AEMET_API_KEY. Las descargas de AEMET quedan pendientes.")
    else:
        aemet_client = AEMETOpenDataClient(
            api_key=aemet_api_key,
            timeout_seconds=HTTP_TIMEOUT_SECONDS,
            wait_seconds=max(AEMET_WAIT_SECONDS, 1.5),
        )

        inventory_endpoint = (
            "https://opendata.aemet.es/opendata/api/"
            "valores/climatologicos/inventarioestaciones/todasestaciones"
        )

        forecast_endpoint = (
            "https://opendata.aemet.es/opendata/api/"
            "prediccion/especifica/municipio/horaria/28079"
        )

        try:
            aemet_client.download_to_file(
                endpoint=inventory_endpoint,
                output_path=config.raw_weather_dir / "aemet_stations_inventory_raw.json",
                force=FORCE_REDOWNLOAD,
            )
        except Exception as error:
            print(f"No se pudo descargar el inventario de AEMET: {error}")

        try:
            aemet_client.download_to_file(
                endpoint=forecast_endpoint,
                output_path=config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
                force=FORCE_REDOWNLOAD,
            )
        except Exception as error:
            print(f"No se pudo descargar la predicción horaria de AEMET: {error}")

        daily_climate_output_path = (
            config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json"
        )

        daily_file_is_valid = json_file_has_records(daily_climate_output_path)

        if daily_climate_output_path.exists() and not daily_file_is_valid:
            print(
                "El archivo AEMET diario existe, pero no contiene registros válidos. "
                "Se elimina para regenerarlo."
            )
            daily_climate_output_path.unlink()

        if daily_climate_output_path.exists() and not FORCE_REDOWNLOAD:
            print(f"Ya existe y contiene registros: {daily_climate_output_path.relative_to(config.project_root)}")
        else:
            all_daily_climate_records = []

            for station_id in selected_aemet_station_ids:
                for start_date, end_date in aemet_daily_ranges_2022:
                    encoded_start = quote(start_date, safe="")
                    encoded_end = quote(end_date, safe="")

                    endpoint = (
                        "https://opendata.aemet.es/opendata/api/"
                        f"valores/climatologicos/diarios/datos/fechaini/{encoded_start}/"
                        f"fechafin/{encoded_end}/estacion/{station_id}"
                    )

                    print(f"Descargando AEMET diario: estación={station_id}, rango={start_date} a {end_date}")

                    try:
                        records = download_aemet_endpoint_json_with_retries(
                            client=aemet_client,
                            endpoint=endpoint,
                            attempts=3,
                            wait_seconds=4.0,
                        )

                        if isinstance(records, list):
                            for record in records:
                                record["_station_id_requested"] = station_id
                                record["_range_start"] = start_date
                                record["_range_end"] = end_date

                            all_daily_climate_records.extend(records)

                        else:
                            all_daily_climate_records.append(
                                {
                                    "_station_id_requested": station_id,
                                    "_range_start": start_date,
                                    "_range_end": end_date,
                                    "_raw_response": records,
                                }
                            )

                    except Exception as error:
                        print(
                            "No se pudo descargar climatología diaria "
                            f"para estación={station_id}, rango={start_date} a {end_date}: {error}"
                        )

            daily_climate_output_path.parent.mkdir(parents=True, exist_ok=True)
            daily_climate_output_path.write_text(
                json.dumps(all_daily_climate_records, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            size_mb = daily_climate_output_path.stat().st_size / 1024 / 1024

            print(
                "Archivo AEMET diario generado: "
                f"{daily_climate_output_path.relative_to(config.project_root)} ({size_mb:.2f} MB)"
            )
            print(f"Registros guardados: {len(all_daily_climate_records)}")

            if len(all_daily_climate_records) == 0:
                print(
                    "Aviso: no se han guardado registros diarios de AEMET. "
                    "El notebook continuará, pero la base puede quedar sin meteorología histórica."
                )


Ya existe: data\raw\weather\aemet_stations_inventory_raw.json
Ya existe: data\raw\weather\aemet_forecast_hourly_madrid_raw.json
Ya existe y contiene registros: data\raw\weather\aemet_daily_climate_selected_stations_2022_raw.json


### 8.6 Verificación posterior a la descarga


In [9]:
raw_catalog_df = RawDatasetCatalog(raw_dataset_specs).to_dataframe()
display(raw_catalog_df)

total_raw_files = len(raw_catalog_df)
existing_raw_files = int(raw_catalog_df["Existe localmente"].sum())
missing_raw_files = total_raw_files - existing_raw_files

total_required_raw_files = int(raw_catalog_df["Necesario para base de modelado"].sum())
existing_required_raw_files = int(
    raw_catalog_df[
        raw_catalog_df["Necesario para base de modelado"]
    ]["Existe localmente"].sum()
)
missing_required_raw_files = total_required_raw_files - existing_required_raw_files

print("Estado general de archivos raw:")
print(f"Archivos raw esperados: {total_raw_files}")
print(f"Archivos raw existentes: {existing_raw_files}")
print(f"Archivos raw pendientes: {missing_raw_files}")

print("\nEstado de archivos raw necesarios para la base de modelado:")
print(f"Archivos necesarios esperados: {total_required_raw_files}")
print(f"Archivos necesarios existentes: {existing_required_raw_files}")
print(f"Archivos necesarios pendientes: {missing_required_raw_files}")

if missing_required_raw_files == 0:
    print("\nTodas las fuentes raw necesarias para la base de modelado existen localmente.")
else:
    print("\nAún faltan fuentes raw necesarias. Deben revisarse antes de limpiar.")


,Dataset,Fuente,URL o endpoint de referencia,Ruta raw esperada,Existe localmente,Tamaño MB,Requiere API Key,Descarga pesada,Necesario para base de modelado,Nota de acceso
0,GBFS estaciones actuales BiciMAD,EMT Madrid / BiciMAD,https://madrid.publicbikesystem.net/customer/g...,data\raw\bicimad\gbfs_station_information_raw....,True,0.51,False,False,False,"Feed GBFS actual. Útil para referencia, mapa o..."
1,GBFS estado actual BiciMAD,EMT Madrid / BiciMAD,https://madrid.publicbikesystem.net/customer/g...,data\raw\bicimad\gbfs_station_status_snapshot_...,True,0.27,False,False,False,Feed GBFS actual. Fotografía de disponibilidad.
2,Histórico diario de viajes BiciMAD,EMT Madrid / BiciMAD,https://datos.emtmadrid.es/dataset/a3442c30-ee...,data\raw\bicimad\bicimad_daily_trips_raw.csv,True,0.01,False,False,False,Serie diaria de viajes. Sirve como contexto de...
3,Histórico de estaciones BiciMAD 2022,EMT Madrid / BiciMAD,https://media.emtmadrid.es/-uHaW6iZkhG,data\raw\bicimad\bicimad_station_status_histor...,True,300.74,False,True,True,Descarga pesada. Se usará la parte de estado d...
4,Calendario laboral de Madrid,Ayuntamiento de Madrid,https://datos.madrid.es/dataset/300082-0-calen...,data\raw\calendar\madrid_working_calendar_raw.csv,True,0.16,False,False,True,Calendario laboral 2013-2026.
5,Inventario de estaciones AEMET,AEMET OpenData,https://opendata.aemet.es/opendata/api/valores...,data\raw\weather\aemet_stations_inventory_raw....,True,0.18,True,False,False,Permite seleccionar estaciones meteorológicas ...
6,Climatología diaria AEMET 2022,AEMET OpenData,https://opendata.aemet.es/opendata/api/valores...,data\raw\weather\aemet_daily_climate_selected_...,True,1.17,True,False,True,Datos meteorológicos diarios observados de 202...
7,Predicción horaria AEMET Madrid,AEMET OpenData,https://opendata.aemet.es/opendata/api/predicc...,data\raw\weather\aemet_forecast_hourly_madrid_...,True,0.04,True,False,False,Predicción horaria para demo futura.


Estado general de archivos raw:
Archivos raw esperados: 8
Archivos raw existentes: 8
Archivos raw pendientes: 0

Estado de archivos raw necesarios para la base de modelado:
Archivos necesarios esperados: 3
Archivos necesarios existentes: 3
Archivos necesarios pendientes: 0

Todas las fuentes raw necesarias para la base de modelado existen localmente.


## 9. Validación inicial de datos raw

En esta sección se validan los archivos originales que existan localmente en `data/raw`.

La validación inicial no transforma los datos. Su objetivo es comprobar que los archivos descargados pueden leerse correctamente y que tienen un formato coherente con lo esperado.


In [10]:
@dataclass
class RawFileValidationResult:
    dataset: str
    path: str
    expected_type: str
    exists: bool
    size_mb: float
    can_be_read: bool
    status: str
    details: str


class RawFileValidator:
    def __init__(self, spec: RawDatasetSpec, project_root: Path):
        self.spec = spec
        self.project_root = project_root
        self.path = spec.local_path

    def detect_expected_type(self) -> str:
        suffix = self.path.suffix.lower()
        if suffix == ".json":
            return "json"
        if suffix == ".csv":
            return "csv"
        if suffix == ".zip":
            return "zip"
        return "unknown"

    def validate(self) -> RawFileValidationResult:
        expected_type = self.detect_expected_type()

        if not self.path.exists():
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=False,
                size_mb=0,
                can_be_read=False,
                status="pendiente",
                details="El archivo no existe localmente.",
            )

        size_mb = round(self.path.stat().st_size / 1024 / 1024, 2)

        if size_mb == 0:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details="El archivo existe pero está vacío.",
            )

        try:
            if expected_type == "json":
                with self.path.open("r", encoding="utf-8") as file:
                    data = json.load(file)
                if isinstance(data, dict):
                    details = f"JSON válido tipo dict. Claves detectadas: {list(data.keys())[:10]}"
                elif isinstance(data, list):
                    details = f"JSON válido tipo list. Elementos detectados: {len(data)}"
                else:
                    details = f"JSON válido. Tipo raíz: {type(data).__name__}"

            elif expected_type == "csv":
                sample = pd.read_csv(self.path, sep=None, engine="python", nrows=5)
                details = f"CSV legible. Columnas detectadas: {list(sample.columns)}."

            elif expected_type == "zip":
                with zipfile.ZipFile(self.path, "r") as zip_file:
                    names = zip_file.namelist()
                details = f"ZIP legible. Archivos internos detectados: {len(names)}. Primeros: {names[:10]}"

            else:
                return RawFileValidationResult(
                    dataset=self.spec.dataset,
                    path=str(self.path.relative_to(self.project_root)),
                    expected_type=expected_type,
                    exists=True,
                    size_mb=size_mb,
                    can_be_read=False,
                    status="revisar",
                    details="Extensión no reconocida.",
                )

            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=True,
                status="ok",
                details=details,
            )

        except Exception as error:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details=str(error),
            )


raw_validation_results = [
    RawFileValidator(spec=spec, project_root=config.project_root).validate()
    for spec in raw_dataset_specs
]

raw_validation_df = pd.DataFrame(
    [
        {
            "Dataset": result.dataset,
            "Ruta": result.path,
            "Tipo esperado": result.expected_type,
            "Existe": result.exists,
            "Tamaño MB": result.size_mb,
            "Se puede leer": result.can_be_read,
            "Estado": result.status,
            "Detalle": result.details,
        }
        for result in raw_validation_results
    ]
)

display(raw_validation_df)

validation_status_summary = (
    raw_validation_df
    .groupby("Estado", dropna=False)
    .agg(archivos=("Dataset", "count"), tamano_total_mb=("Tamaño MB", "sum"))
    .reset_index()
)
display(validation_status_summary)


,Dataset,Ruta,Tipo esperado,Existe,Tamaño MB,Se puede leer,Estado,Detalle
0,GBFS estaciones actuales BiciMAD,data\raw\bicimad\gbfs_station_information_raw....,json,True,0.51,True,ok,JSON válido tipo dict. Claves detectadas: ['la...
1,GBFS estado actual BiciMAD,data\raw\bicimad\gbfs_station_status_snapshot_...,json,True,0.27,True,ok,JSON válido tipo dict. Claves detectadas: ['la...
2,Histórico diario de viajes BiciMAD,data\raw\bicimad\bicimad_daily_trips_raw.csv,csv,True,0.01,True,ok,CSV legible. Columnas detectadas: ['\ufeffFECH...
3,Histórico de estaciones BiciMAD 2022,data\raw\bicimad\bicimad_station_status_histor...,zip,True,300.74,True,ok,ZIP legible. Archivos internos detectados: 49....
4,Calendario laboral de Madrid,data\raw\calendar\madrid_working_calendar_raw.csv,csv,True,0.16,True,ok,CSV legible. Columnas detectadas: ['\ufeffDia'...
5,Inventario de estaciones AEMET,data\raw\weather\aemet_stations_inventory_raw....,json,True,0.18,True,ok,JSON válido tipo list. Elementos detectados: 920
6,Climatología diaria AEMET 2022,data\raw\weather\aemet_daily_climate_selected_...,json,True,1.17,True,ok,JSON válido tipo list. Elementos detectados: 1702
7,Predicción horaria AEMET Madrid,data\raw\weather\aemet_forecast_hourly_madrid_...,json,True,0.04,True,ok,JSON válido tipo list. Elementos detectados: 1


,Estado,archivos,tamano_total_mb
0,ok,8,303.08


## 10. Limpieza de datasets auxiliares

En esta sección se limpian datasets auxiliares del proyecto.

Estos datasets no sustituyen al histórico principal de estaciones BiciMAD 2022, pero ayudan a completar el contexto del proyecto.

### 10.1 GBFS estaciones actuales


In [11]:
def safe_read_json(path: Path):
    if not path.exists():
        print(f"No existe: {path.relative_to(config.project_root)}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def extract_gbfs_stations(raw_data: dict) -> list[dict]:
    if raw_data is None:
        return []
    if isinstance(raw_data, dict) and "data" in raw_data and isinstance(raw_data["data"], dict):
        if "stations" in raw_data["data"]:
            return raw_data["data"]["stations"]
    if isinstance(raw_data, dict) and "stations" in raw_data:
        return raw_data["stations"]
    return []


station_information_raw_path = config.raw_bicimad_dir / "gbfs_station_information_raw.json"
station_information_output_path = config.interim_bicimad_dir / "stations_clean.csv"

station_information_raw = safe_read_json(station_information_raw_path)
station_information_records = extract_gbfs_stations(station_information_raw)

if not station_information_records:
    stations_clean_df = pd.DataFrame()
    print("No hay registros de estaciones actuales para limpiar.")
else:
    df = pd.json_normalize(station_information_records)

    rename_map = {
        "station_id": "station_id_current",
        "name": "station_name",
        "lat": "station_latitude",
        "lon": "station_longitude",
        "capacity": "station_capacity",
        "address": "station_address",
    }
    df = df.rename(columns=rename_map)

    expected_columns = [
        "station_id_current",
        "station_name",
        "station_latitude",
        "station_longitude",
        "station_capacity",
        "station_address",
    ]

    for column in expected_columns:
        if column not in df.columns:
            df[column] = pd.NA

    stations_clean_df = df[expected_columns].copy()

    for column in ["station_latitude", "station_longitude", "station_capacity"]:
        stations_clean_df[column] = pd.to_numeric(stations_clean_df[column], errors="coerce")

    stations_clean_df["station_id_current"] = stations_clean_df["station_id_current"].astype("string")
    stations_clean_df["station_name"] = stations_clean_df["station_name"].astype("string")
    stations_clean_df["station_address"] = stations_clean_df["station_address"].astype("string")

    stations_clean_df = stations_clean_df.sort_values("station_id_current").reset_index(drop=True)
    stations_clean_df.to_csv(station_information_output_path, index=False)

    print(f"Archivo generado: {station_information_output_path.relative_to(config.project_root)}")
    print(f"Filas: {len(stations_clean_df)}")
    display(stations_clean_df.head())


Archivo generado: data\interim\bicimad\stations_clean.csv
Filas: 653


,station_id_current,station_name,station_latitude,station_longitude,station_capacity,station_address
0,1406,2 - Metro Callao,40.420400,-3.705690,47,Calle Miguel Moya nº 1
1,1407,3 - Plaza Conde Suchil,40.430322,-3.707254,39,"Plaza Conde Surchill, 4"
2,1408,4 - Malasaña,40.428626,-3.702500,47,Calle Manuela Malasaña nº 3
3,1409,5 - Fuencarral,40.428521,-3.702135,47,Calle Fuencarral nº 106
4,1410,6 - Colegio de Arquitectos,40.423178,-3.699023,19,"Calle de Hortaleza, 53"


### 10.2 GBFS estado actual


In [12]:
station_status_raw_path = config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json"
station_status_output_path = config.interim_bicimad_dir / "station_status_snapshot_clean.csv"

station_status_raw = safe_read_json(station_status_raw_path)
station_status_records = extract_gbfs_stations(station_status_raw)

if not station_status_records:
    station_status_snapshot_clean_df = pd.DataFrame()
    print("No hay registros de estado actual para limpiar.")
else:
    df = pd.json_normalize(station_status_records)

    expected_columns = [
        "station_id",
        "num_bikes_available",
        "num_docks_available",
        "num_bikes_disabled",
        "num_docks_disabled",
        "is_installed",
        "is_renting",
        "is_returning",
        "last_reported",
    ]

    for column in expected_columns:
        if column not in df.columns:
            df[column] = pd.NA

    station_status_snapshot_clean_df = df[expected_columns].copy()
    station_status_snapshot_clean_df = station_status_snapshot_clean_df.rename(
        columns={"station_id": "station_id_current"}
    )

    station_status_snapshot_clean_df["station_id_current"] = (
        station_status_snapshot_clean_df["station_id_current"].astype("string")
    )

    numeric_columns = [
        "num_bikes_available",
        "num_docks_available",
        "num_bikes_disabled",
        "num_docks_disabled",
        "is_installed",
        "is_renting",
        "is_returning",
        "last_reported",
    ]

    for column in numeric_columns:
        station_status_snapshot_clean_df[column] = pd.to_numeric(
            station_status_snapshot_clean_df[column],
            errors="coerce",
        )

    station_status_snapshot_clean_df["last_reported_datetime"] = pd.to_datetime(
        station_status_snapshot_clean_df["last_reported"],
        unit="s",
        errors="coerce",
    )

    station_status_snapshot_clean_df["station_available_for_renting_and_returning"] = (
        (station_status_snapshot_clean_df["is_installed"] == 1)
        & (station_status_snapshot_clean_df["is_renting"] == 1)
        & (station_status_snapshot_clean_df["is_returning"] == 1)
    )

    station_status_snapshot_clean_df = station_status_snapshot_clean_df.sort_values("station_id_current").reset_index(drop=True)
    station_status_snapshot_clean_df.to_csv(station_status_output_path, index=False)

    print(f"Archivo generado: {station_status_output_path.relative_to(config.project_root)}")
    print(f"Filas: {len(station_status_snapshot_clean_df)}")
    display(station_status_snapshot_clean_df.head())


Archivo generado: data\interim\bicimad\station_status_snapshot_clean.csv
Filas: 653


,station_id_current,num_bikes_available,num_docks_available,num_bikes_disabled,num_docks_disabled,is_installed,is_renting,is_returning,last_reported,last_reported_datetime,station_available_for_renting_and_returning
0,1406,0,22,5,0,True,True,True,1782915421,2026-07-01 14:17:01,True
1,1407,5,12,2,0,True,True,True,1782915415,2026-07-01 14:16:55,True
2,1408,0,27,0,0,True,True,True,1782915419,2026-07-01 14:16:59,True
3,1409,5,20,0,1,True,True,True,1782915293,2026-07-01 14:14:53,True
4,1410,0,19,0,0,True,False,False,1782915374,2026-07-01 14:16:14,False


### 10.3 Histórico diario de viajes


In [13]:
daily_trips_raw_path = config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv"
daily_trips_output_path = config.interim_bicimad_dir / "bicimad_daily_trips_clean.csv"

if not daily_trips_raw_path.exists():
    daily_trips_clean_df = pd.DataFrame()
    print(f"No existe: {daily_trips_raw_path.relative_to(config.project_root)}")
else:
    daily_trips_raw_df = pd.read_csv(daily_trips_raw_path, sep=None, engine="python")
    daily_trips_raw_df.columns = [str(col).strip().upper() for col in daily_trips_raw_df.columns]

    date_col = "FECHA" if "FECHA" in daily_trips_raw_df.columns else daily_trips_raw_df.columns[0]
    trips_col = "VIAJES" if "VIAJES" in daily_trips_raw_df.columns else daily_trips_raw_df.columns[1]

    daily_trips_clean_df = daily_trips_raw_df[[date_col, trips_col]].copy()
    daily_trips_clean_df = daily_trips_clean_df.rename(
        columns={date_col: "date", trips_col: "daily_trips"}
    )

    daily_trips_clean_df["date"] = pd.to_datetime(
        daily_trips_clean_df["date"].astype(str),
        format="%Y%m%d",
        errors="coerce",
    ).fillna(
        pd.to_datetime(daily_trips_clean_df["date"], errors="coerce", dayfirst=True)
    )

    daily_trips_clean_df["daily_trips"] = pd.to_numeric(
        daily_trips_clean_df["daily_trips"],
        errors="coerce",
    )

    daily_trips_clean_df = (
        daily_trips_clean_df
        .dropna(subset=["date"])
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    daily_trips_clean_df.to_csv(daily_trips_output_path, index=False)

    print(f"Archivo generado: {daily_trips_output_path.relative_to(config.project_root)}")
    print(f"Rango de fechas: {daily_trips_clean_df['date'].min()} → {daily_trips_clean_df['date'].max()}")
    print(f"Filas: {len(daily_trips_clean_df)}")
    display(daily_trips_clean_df.head())


Archivo generado: data\interim\bicimad\bicimad_daily_trips_clean.csv
Rango de fechas: 2024-02-01 00:00:00 → 2026-04-30 00:00:00
Filas: 820


,date,daily_trips
0,2024-02-01,26194
1,2024-02-02,24706
2,2024-02-03,21059
3,2024-02-04,19990
4,2024-02-05,21616


### 10.4 Calendario laboral


In [14]:
calendar_raw_path = config.raw_calendar_dir / "madrid_working_calendar_raw.csv"
calendar_output_path = config.interim_calendar_dir / "madrid_working_calendar_clean.csv"

def normalize_colname(value: str) -> str:
    value = str(value).strip().lower()
    replacements = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
        "ñ": "n", " ": "_", "-": "_", ".": "_"
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def find_best_date_column(df: pd.DataFrame) -> str:
    best_column = None
    best_score = -1

    for column in df.columns:
        parsed = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
        score = parsed.notna().sum()

        if score > best_score:
            best_score = score
            best_column = column

    if best_column is None or best_score == 0:
        raise ValueError("No se ha podido detectar una columna de fecha en el calendario.")

    return best_column


if not calendar_raw_path.exists():
    calendar_clean_df = pd.DataFrame()
    print(f"No existe: {calendar_raw_path.relative_to(config.project_root)}")
else:
    calendar_raw_df = pd.read_csv(
        calendar_raw_path,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
    )

    calendar_raw_df.columns = [normalize_colname(col) for col in calendar_raw_df.columns]

    date_col = find_best_date_column(calendar_raw_df)

    calendar_clean_df = pd.DataFrame()
    calendar_clean_df["date"] = pd.to_datetime(
        calendar_raw_df[date_col],
        errors="coerce",
        dayfirst=True,
    )

    calendar_clean_df = calendar_clean_df.dropna(subset=["date"]).copy()

    text_context = (
        calendar_raw_df
        .reindex(calendar_clean_df.index)
        .astype("string")
        .fillna("")
        .apply(lambda row: " ".join([str(value) for value in row.to_list()]).lower(), axis=1)
    )

    day_name_map_es = {
        0: "lunes",
        1: "martes",
        2: "miércoles",
        3: "jueves",
        4: "viernes",
        5: "sábado",
        6: "domingo",
    }

    calendar_clean_df["year"] = calendar_clean_df["date"].dt.year
    calendar_clean_df["month"] = calendar_clean_df["date"].dt.month
    calendar_clean_df["day_of_week"] = calendar_clean_df["date"].dt.dayofweek
    calendar_clean_df["day_of_week_name_es"] = calendar_clean_df["day_of_week"].map(day_name_map_es)

    calendar_clean_df["is_weekend"] = calendar_clean_df["day_of_week"].isin([5, 6])
    calendar_clean_df["is_holiday"] = text_context.str.contains("festivo|fiesta|inhabil", regex=True, na=False)

    calendar_clean_df["day_type"] = np.select(
        [
            calendar_clean_df["is_holiday"],
            calendar_clean_df["day_of_week"] == 5,
            calendar_clean_df["day_of_week"] == 6,
        ],
        [
            "festivo",
            "sabado",
            "domingo",
        ],
        default="laborable",
    )

    calendar_clean_df["is_working_day"] = calendar_clean_df["day_type"].eq("laborable")
    calendar_clean_df["is_non_working_day"] = ~calendar_clean_df["is_working_day"]
    calendar_clean_df["holiday_type"] = np.where(calendar_clean_df["is_holiday"], "festivo", pd.NA)
    calendar_clean_df["holiday_name"] = pd.NA

    calendar_clean_df = (
        calendar_clean_df
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    calendar_clean_df.to_csv(calendar_output_path, index=False)

    print(f"Archivo generado: {calendar_output_path.relative_to(config.project_root)}")
    print(f"Rango de fechas: {calendar_clean_df['date'].min()} → {calendar_clean_df['date'].max()}")
    print(f"Filas: {len(calendar_clean_df)}")
    display(calendar_clean_df.head())


C:\Users\elena\AppData\Local\Temp\ipykernel_4240\2542761137.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
C:\Users\elena\AppData\Local\Temp\ipykernel_4240\2542761137.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
C:\Users\elena\AppData\Local\Temp\ipykernel_4240\2542761137.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
C:\Users\elena\AppData\Local\Temp\ipy

Archivo generado: data\interim\calendar\madrid_working_calendar_clean.csv
Rango de fechas: 2013-01-01 00:00:00 → 2026-12-31 00:00:00
Filas: 5112


,date,year,month,day_of_week,day_of_week_name_es,is_weekend,is_holiday,day_type,is_working_day,is_non_working_day,holiday_type,holiday_name
0,2013-01-01,2013,1,1,martes,False,True,festivo,False,True,festivo,<NA>
1,2013-01-02,2013,1,2,miércoles,False,False,laborable,True,False,NaN,<NA>
2,2013-01-03,2013,1,3,jueves,False,False,laborable,True,False,NaN,<NA>
3,2013-01-04,2013,1,4,viernes,False,False,laborable,True,False,NaN,<NA>
4,2013-01-05,2013,1,5,sábado,True,False,sabado,False,True,NaN,<NA>


### 10.5 Meteorología AEMET


In [15]:
def to_float_es(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", ".")

    if text in {"", "nan", "None"}:
        return np.nan

    if text.lower() in {"ip", "tr", "trace"}:
        return 0.0

    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return np.nan

    return float(match.group(0))


# Inventario AEMET
aemet_inventory_raw_path = config.raw_weather_dir / "aemet_stations_inventory_raw.json"
aemet_inventory_output_path = config.interim_weather_dir / "aemet_stations_inventory_clean.csv"

inventory_raw = safe_read_json(aemet_inventory_raw_path)

if isinstance(inventory_raw, list):
    inventory_df = pd.DataFrame(inventory_raw)
elif isinstance(inventory_raw, dict):
    inventory_df = pd.json_normalize(inventory_raw)
else:
    inventory_df = pd.DataFrame()

if inventory_df.empty:
    aemet_inventory_clean_df = pd.DataFrame()
    print("No hay inventario AEMET para limpiar.")
else:
    aemet_inventory_clean_df = inventory_df.copy()
    aemet_inventory_clean_df.columns = [normalize_colname(col) for col in aemet_inventory_clean_df.columns]
    aemet_inventory_clean_df.to_csv(aemet_inventory_output_path, index=False)
    print(f"Archivo generado: {aemet_inventory_output_path.relative_to(config.project_root)}")
    display(aemet_inventory_clean_df.head())


# Climatología diaria AEMET 2022
aemet_daily_raw_path = config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json"
aemet_daily_output_path = config.interim_weather_dir / "aemet_daily_climate_selected_stations_2022_clean.csv"
aemet_daily_join_ready_path = config.interim_weather_dir / "aemet_daily_weather_madrid_2022_join_ready.csv"

daily_weather_raw = safe_read_json(aemet_daily_raw_path)

if isinstance(daily_weather_raw, list):
    daily_weather_df = pd.DataFrame(daily_weather_raw)
else:
    daily_weather_df = pd.DataFrame()

if daily_weather_df.empty:
    aemet_daily_weather_madrid_2022_join_ready_df = pd.DataFrame()
    print("No hay datos diarios AEMET para limpiar.")
else:
    daily_weather_df.columns = [normalize_colname(col) for col in daily_weather_df.columns]

    date_col = "fecha" if "fecha" in daily_weather_df.columns else None
    station_col = "indicativo" if "indicativo" in daily_weather_df.columns else "_station_id_requested"

    if date_col is None:
        raise ValueError("No se ha encontrado columna de fecha en AEMET diario.")

    aemet_daily_clean_df = pd.DataFrame()
    aemet_daily_clean_df["date"] = pd.to_datetime(daily_weather_df[date_col], errors="coerce")
    aemet_daily_clean_df["weather_station_id"] = daily_weather_df[station_col].astype("string")

    source_map = {
        "weather_temperature_mean_c": ["tmed", "temperatura_media"],
        "weather_temperature_min_c": ["tmin", "temperatura_minima"],
        "weather_temperature_max_c": ["tmax", "temperatura_maxima"],
        "weather_precipitation_mm": ["prec", "precipitacion"],
        "weather_humidity_mean_pct": ["hrmedia", "humedad_media"],
    }

    for target_col, possible_cols in source_map.items():
        source_col = next((col for col in possible_cols if col in daily_weather_df.columns), None)
        if source_col is None:
            aemet_daily_clean_df[target_col] = np.nan
        else:
            aemet_daily_clean_df[target_col] = daily_weather_df[source_col].apply(to_float_es)

    aemet_daily_clean_df = aemet_daily_clean_df.dropna(subset=["date"]).copy()
    aemet_daily_clean_df["weather_has_precipitation"] = aemet_daily_clean_df["weather_precipitation_mm"].fillna(0) > 0

    aemet_daily_clean_df.to_csv(aemet_daily_output_path, index=False)

    aemet_daily_weather_madrid_2022_join_ready_df = (
        aemet_daily_clean_df
        .groupby("date", as_index=False)
        .agg(
            weather_temperature_mean_c=("weather_temperature_mean_c", "mean"),
            weather_temperature_min_c=("weather_temperature_min_c", "mean"),
            weather_temperature_max_c=("weather_temperature_max_c", "mean"),
            weather_precipitation_mm=("weather_precipitation_mm", "mean"),
            weather_humidity_mean_pct=("weather_humidity_mean_pct", "mean"),
            weather_station_count=("weather_station_id", "nunique"),
            weather_temperature_available_count=("weather_temperature_mean_c", lambda s: int(s.notna().sum())),
            weather_precipitation_available_count=("weather_precipitation_mm", lambda s: int(s.notna().sum())),
            weather_humidity_available_count=("weather_humidity_mean_pct", lambda s: int(s.notna().sum())),
        )
    )

    aemet_daily_weather_madrid_2022_join_ready_df["weather_has_precipitation"] = (
        aemet_daily_weather_madrid_2022_join_ready_df["weather_precipitation_mm"].fillna(0) > 0
    )

    aemet_daily_weather_madrid_2022_join_ready_df = (
        aemet_daily_weather_madrid_2022_join_ready_df
        .sort_values("date")
        .reset_index(drop=True)
    )

    aemet_daily_weather_madrid_2022_join_ready_df.to_csv(aemet_daily_join_ready_path, index=False)

    print(f"Archivo generado: {aemet_daily_output_path.relative_to(config.project_root)}")
    print(f"Archivo generado: {aemet_daily_join_ready_path.relative_to(config.project_root)}")
    print(f"Rango de fechas: {aemet_daily_weather_madrid_2022_join_ready_df['date'].min()} → {aemet_daily_weather_madrid_2022_join_ready_df['date'].max()}")
    display(aemet_daily_weather_madrid_2022_join_ready_df.head())


# Predicción horaria AEMET para demo futura
forecast_raw_path = config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json"
forecast_output_path = config.interim_weather_dir / "aemet_forecast_hourly_madrid_clean.csv"
forecast_raw = safe_read_json(forecast_raw_path)

forecast_records = []

try:
    if isinstance(forecast_raw, list) and forecast_raw:
        prediccion = forecast_raw[0].get("prediccion", {})
        dias = prediccion.get("dia", [])

        for dia in dias:
            fecha = dia.get("fecha")
            for variable_group in ["temperatura", "humedadRelativa", "precipitacion"]:
                for item in dia.get(variable_group, []):
                    forecast_records.append(
                        {
                            "forecast_date": fecha,
                            "hour": item.get("periodo"),
                            "variable": variable_group,
                            "value": item.get("value"),
                        }
                    )

    aemet_forecast_hourly_madrid_clean_df = pd.DataFrame(forecast_records)

    if not aemet_forecast_hourly_madrid_clean_df.empty:
        aemet_forecast_hourly_madrid_clean_df.to_csv(forecast_output_path, index=False)
        print(f"Archivo generado: {forecast_output_path.relative_to(config.project_root)}")
        display(aemet_forecast_hourly_madrid_clean_df.head())
    else:
        print("No se han encontrado registros horarios en la predicción AEMET.")
except Exception as error:
    print(f"No se pudo limpiar la predicción horaria AEMET: {error}")


Archivo generado: data\interim\weather\aemet_stations_inventory_clean.csv


,latitud,provincia,altitud,indicativo,nombre,indsinop,longitud
0,394924N,ILLES BALEARS,490,B013X,"ESCORCA, LLUC",08304,025309E
1,394744N,BALEARES,5,B051A,"SÓLLER, PUERTO",08316,024129E
2,394121N,ILLES BALEARS,60,B087X,BANYALBUFAR,,023046E
3,393446N,BALEARES,52,B103B,ANDRATX - SANT ELM,,022208E
4,393305N,BALEARES,50,B158X,"CALVIÀ, ES CAPDELLÀ",,022759E


Archivo generado: data\interim\weather\aemet_daily_climate_selected_stations_2022_clean.csv
Archivo generado: data\interim\weather\aemet_daily_weather_madrid_2022_join_ready.csv
Rango de fechas: 2022-01-01 00:00:00 → 2022-12-31 00:00:00


,date,weather_temperature_mean_c,weather_temperature_min_c,weather_temperature_max_c,weather_precipitation_mm,weather_humidity_mean_pct,weather_station_count,weather_temperature_available_count,weather_precipitation_available_count,weather_humidity_available_count,weather_has_precipitation
0,2022-01-01,10.06,2.36,17.76,0.00,63.6,5,5,5,5,False
1,2022-01-02,8.86,2.66,15.06,0.00,73.8,5,5,5,5,False
2,2022-01-03,8.92,3.20,14.64,0.00,77.4,5,5,5,5,False
3,2022-01-04,7.58,2.88,12.30,6.20,86.6,5,5,5,5,True
4,2022-01-05,6.44,3.40,9.50,5.34,78.0,5,5,5,5,True


Archivo generado: data\interim\weather\aemet_forecast_hourly_madrid_clean.csv


,forecast_date,hour,variable,value
0,2026-07-01T00:00:00,09,temperatura,24
1,2026-07-01T00:00:00,10,temperatura,26
2,2026-07-01T00:00:00,11,temperatura,28
3,2026-07-01T00:00:00,12,temperatura,30
4,2026-07-01T00:00:00,13,temperatura,32


## 11. Limpieza del histórico principal BiciMAD 2022

Esta sección limpia el histórico de estado de estaciones de BiciMAD 2022.

Es la fuente principal del proyecto. El resultado será:

`data/interim/bicimad/station_status_history_2022_clean.csv`

La limpieza se ha diseñado para ser robusta ante distintas estructuras internas del ZIP histórico.


In [16]:
historical_raw_zip_path = config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip"
historical_clean_output_path = config.interim_bicimad_dir / "station_status_history_2022_clean.csv"
historical_zip_inventory_debug_path = config.interim_bicimad_dir / "historical_zip_inventory_debug.csv"

historical_output_columns = [
    "source_month",
    "snapshot_datetime",
    "station_id_historical",
    "station_number",
    "station_name",
    "station_address",
    "station_latitude",
    "station_longitude",
    "station_capacity",
    "num_docks_available",
    "num_bikes_available",
    "reservations_count",
    "station_light_status",
    "is_active",
    "is_not_available",
    "occupation_ratio",
]


def safe_decode_json_bytes(raw_bytes: bytes):
    for encoding in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            text = raw_bytes.decode(encoding)
            break
        except UnicodeDecodeError:
            text = None

    if text is None:
        return None

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        records = []

        for line in text.splitlines():
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

        if records:
            return records

    return None


def iter_files_recursively_from_zip_bytes(raw_zip_bytes: bytes, prefix: str = ""):
    try:
        with zipfile.ZipFile(BytesIO(raw_zip_bytes), "r") as zip_file:
            for zip_info in zip_file.infolist():
                if zip_info.is_dir():
                    continue

                file_name = zip_info.filename
                full_name = f"{prefix}/{file_name}" if prefix else file_name
                lower_name = file_name.lower()

                try:
                    file_bytes = zip_file.read(file_name)
                except Exception:
                    continue

                if lower_name.endswith(".zip"):
                    yield from iter_files_recursively_from_zip_bytes(file_bytes, prefix=full_name)
                else:
                    yield full_name, zip_info, file_bytes

    except zipfile.BadZipFile:
        return


def iter_files_recursively_from_zip_path(zip_path: Path):
    with zip_path.open("rb") as file:
        raw_zip_bytes = file.read()

    yield from iter_files_recursively_from_zip_bytes(raw_zip_bytes)


def get_nested_value(record: dict, keys: list[str], default=np.nan):
    for key in keys:
        if "." in key:
            current = record
            ok = True

            for part in key.split("."):
                if isinstance(current, dict) and part in current:
                    current = current[part]
                else:
                    ok = False
                    break

            if ok:
                return current

        elif isinstance(record, dict) and key in record:
            return record[key]

    return default


def extract_coordinates(record: dict):
    lat = get_nested_value(record, ["lat", "latitude", "station_latitude"], np.nan)
    lon = get_nested_value(record, ["lon", "lng", "longitude", "station_longitude"], np.nan)

    geometry = record.get("geometry") if isinstance(record, dict) else None

    if isinstance(geometry, dict):
        coords = geometry.get("coordinates")

        if isinstance(coords, list) and len(coords) >= 2:
            lon = coords[0]
            lat = coords[1]

    return lat, lon


def parse_snapshot_datetime(root, file_name: str, zip_info=None):
    """
    Extrae la fecha/hora del snapshot de una estación BiciMAD.
    
    El JSON histórico de EMT usa '_id' como campo de fecha (ISO 8601).
    Ejemplo: "2022-01-01T00:13:20.603583"
    
    Prioridad:
    1. _id (campo específico del histórico EMT 2022)
    2. last_updated, datetime, timestamp, date, fecha (campos GBFS estándar)
    3. Nombre del archivo (fallback)
    4. NaT (nunca usar zip_info.date_time — es fecha de compresión)
    """
    candidates = []

    if isinstance(root, dict):
        # Campo específico del histórico EMT 2022
        candidates.append(root.get("_id"))
        
        # Campos GBFS estándar
        candidates.extend([
            root.get("last_updated"),
            root.get("datetime"),
            root.get("timestamp"),
            root.get("date"),
            root.get("fecha"),
        ])
        
        # Buscar en data anidado
        if isinstance(root.get("data"), dict):
            data = root["data"]
            candidates.append(data.get("_id"))
            candidates.extend([
                data.get("last_updated"),
                data.get("datetime"),
                data.get("timestamp"),
                data.get("date"),
                data.get("fecha"),
            ])

    # 1. Intentar parsear metadatos JSON
    for value in candidates:
        if value is None:
            continue
        if isinstance(value, (int, float)) or str(value).isdigit():
            parsed = pd.to_datetime(value, unit="s", errors="coerce")
        else:
            parsed = pd.to_datetime(value, errors="coerce")
        if pd.notna(parsed):
            return parsed

    # 2. Extraer fecha del nombre del archivo (fallback)
    # Patrones: 202201-json.zip, trips_22_01_January-csv.zip
    patterns = [
        r"(20\d{2})(\d{2})-json",           # 202201-json.zip
        r"trips_(\d{2})_(\d{2})_[A-Za-z]",  # trips_22_01_January-csv.zip
    ]
    
    for pattern in patterns:
        match = re.search(pattern, file_name)
        if match:
            if pattern.startswith(r"(20\d{2})"):
                year, month = match.group(1), match.group(2)
            else:
                year_short, month = match.group(1), match.group(2)
                year = "20" + year_short
            
            parsed = pd.to_datetime(f"{year}-{month}-01", errors="coerce")
            if pd.notna(parsed):
                return parsed

    # 3. Buscar cualquier fecha en el nombre del archivo
    date_match = re.search(r"(20\d{2})[-_]?(\d{2})[-_]?(\d{2})", file_name)
    if date_match:
        year, month, day = date_match.groups()
        parsed = pd.to_datetime(f"{year}-{month}-{day}", errors="coerce")
        if pd.notna(parsed):
            return parsed

    # 4. NUNCA usar zip_info.date_time
    return pd.NaT


def month_from_name(name: str) -> str:
    match = re.search(r"(20\d{2})[-_]?([01]\d)", name)

    if match:
        return f"{match.group(1)}-{match.group(2)}"

    return "unknown"


def station_candidate_score(record: dict) -> int:
    if not isinstance(record, dict):
        return 0

    station_keys = {
        "id",
        "station_id",
        "number",
        "name",
        "address",
        "geometry",
        "dock_bikes",
        "free_bases",
        "total_bases",
        "reservations_count",
        "light",
        "activate",
        "no_available",
        "num_bikes_available",
        "num_docks_available",
        "capacity",
    }

    return len(station_keys.intersection(set(record.keys())))


def looks_like_station_status_records(records) -> bool:
    if not isinstance(records, list) or not records:
        return False

    sample = [record for record in records[:20] if isinstance(record, dict)]

    if not sample:
        return False

    score = sum(station_candidate_score(record) for record in sample) / len(sample)

    availability_keys = {
        "dock_bikes",
        "free_bases",
        "num_bikes_available",
        "num_docks_available",
        "total_bases",
        "capacity",
    }

    has_availability = any(
        bool(availability_keys.intersection(set(record.keys())))
        for record in sample
    )

    has_station_identity = any(
        {"id", "station_id", "number", "name"}.intersection(set(record.keys()))
        for record in sample
    )

    return score >= 3 and has_availability and has_station_identity


def find_station_status_lists(payload):
    found_lists = []

    def walk(obj):
        if isinstance(obj, list):
            if looks_like_station_status_records(obj):
                found_lists.append(obj)
                return

            for item in obj:
                walk(item)

        elif isinstance(obj, dict):
            for value in obj.values():
                walk(value)

    walk(payload)

    return found_lists


def to_numeric_or_nan(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", ".")

    if text == "" or text.lower() in {"nan", "none", "null"}:
        return np.nan

    match = re.search(r"-?\d+(?:\.\d+)?", text)

    if not match:
        return np.nan

    return float(match.group(0))


def to_int_flag(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return int(value)

    text = str(value).strip().lower()

    if text in {"1", "true", "t", "yes", "y", "si", "sí"}:
        return 1

    if text in {"0", "false", "f", "no", "n"}:
        return 0

    try:
        return int(float(text))
    except Exception:
        return np.nan


def normalize_historical_station_record(record: dict, snapshot_datetime, source_month: str):
    lat, lon = extract_coordinates(record)

    station_capacity = get_nested_value(
        record,
        ["capacity", "total_bases", "station_capacity"],
        np.nan,
    )

    num_bikes_available = get_nested_value(
        record,
        ["num_bikes_available", "dock_bikes", "bikes_available"],
        np.nan,
    )

    num_docks_available = get_nested_value(
        record,
        ["num_docks_available", "free_bases", "docks_available"],
        np.nan,
    )

    station_light_status = get_nested_value(
        record,
        ["light", "station_light_status"],
        np.nan,
    )

    is_active = get_nested_value(
        record,
        ["activate", "is_active", "is_installed"],
        np.nan,
    )

    is_not_available = get_nested_value(
        record,
        ["no_available", "is_not_available"],
        np.nan,
    )

    station_light_status = to_numeric_or_nan(station_light_status)
    is_not_available = to_int_flag(is_not_available)

    if pd.isna(is_not_available) and station_light_status == 3:
        is_not_available = 1

    return {
        "source_month": source_month,
        "snapshot_datetime": snapshot_datetime,
        "station_id_historical": get_nested_value(record, ["id", "station_id", "station_id_historical"], np.nan),
        "station_number": get_nested_value(record, ["number", "station_number"], np.nan),
        "station_name": get_nested_value(record, ["name", "station_name"], pd.NA),
        "station_address": get_nested_value(record, ["address", "station_address"], pd.NA),
        "station_latitude": lat,
        "station_longitude": lon,
        "station_capacity": station_capacity,
        "num_docks_available": num_docks_available,
        "num_bikes_available": num_bikes_available,
        "reservations_count": get_nested_value(record, ["reservations_count", "reservation_count"], np.nan),
        "station_light_status": station_light_status,
        "is_active": to_int_flag(is_active),
        "is_not_available": is_not_available,
    }


if not historical_raw_zip_path.exists():
    station_status_history_2022_clean_df = pd.DataFrame(columns=historical_output_columns)
    print(f"No existe: {historical_raw_zip_path.relative_to(config.project_root)}")
else:
    print("Procesando ZIP histórico. Esta celda puede tardar varios minutos.")

    historical_clean_output_path.parent.mkdir(parents=True, exist_ok=True)

    if historical_clean_output_path.exists():
        historical_clean_output_path.unlink()

    inventory_records = []
    buffer = []
    total_records = 0
    processed_json_files = 0
    json_files_seen = 0
    written_header = False
    chunk_size = 100_000

    for file_name, zip_info, raw_bytes in iter_files_recursively_from_zip_path(historical_raw_zip_path):
        lower_name = file_name.lower()

        inventory_records.append(
            {
                "file_name": file_name,
                "size_bytes": len(raw_bytes),
                "is_json": lower_name.endswith(".json"),
            }
        )

        if not lower_name.endswith(".json"):
            continue

        json_files_seen += 1
        payload = safe_decode_json_bytes(raw_bytes)

        if payload is None:
            continue

        station_lists = find_station_status_lists(payload)

        if not station_lists:
            continue

        snapshot_datetime = parse_snapshot_datetime(payload, file_name, zip_info=zip_info)
        source_month = month_from_name(file_name)

        for station_records in station_lists:
            processed_json_files += 1

            for record in station_records:
                normalized = normalize_historical_station_record(
                    record=record,
                    snapshot_datetime=snapshot_datetime,
                    source_month=source_month,
                )
                buffer.append(normalized)

        if len(buffer) >= chunk_size:
            chunk_df = pd.DataFrame(buffer)

            for column in historical_output_columns:
                if column not in chunk_df.columns:
                    chunk_df[column] = pd.NA

            chunk_df = chunk_df[historical_output_columns[:-1]].copy()

            chunk_df["station_capacity"] = pd.to_numeric(chunk_df["station_capacity"], errors="coerce")
            chunk_df["num_bikes_available"] = pd.to_numeric(chunk_df["num_bikes_available"], errors="coerce")
            chunk_df["occupation_ratio"] = (
                chunk_df["num_bikes_available"]
                / chunk_df["station_capacity"].replace({0: np.nan})
            ).clip(0, 1)

            chunk_df = chunk_df[historical_output_columns]

            chunk_df.to_csv(
                historical_clean_output_path,
                index=False,
                mode="a",
                header=not written_header,
            )

            written_header = True
            total_records += len(chunk_df)
            buffer = []

            print(f"Registros escritos: {total_records:,}")

    inventory_df = pd.DataFrame(inventory_records)
    inventory_df.to_csv(historical_zip_inventory_debug_path, index=False)

    if buffer:
        chunk_df = pd.DataFrame(buffer)

        for column in historical_output_columns:
            if column not in chunk_df.columns:
                chunk_df[column] = pd.NA

        chunk_df = chunk_df[historical_output_columns[:-1]].copy()

        chunk_df["station_capacity"] = pd.to_numeric(chunk_df["station_capacity"], errors="coerce")
        chunk_df["num_bikes_available"] = pd.to_numeric(chunk_df["num_bikes_available"], errors="coerce")
        chunk_df["occupation_ratio"] = (
            chunk_df["num_bikes_available"]
            / chunk_df["station_capacity"].replace({0: np.nan})
        ).clip(0, 1)

        chunk_df = chunk_df[historical_output_columns]

        chunk_df.to_csv(
            historical_clean_output_path,
            index=False,
            mode="a",
            header=not written_header,
        )

        written_header = True
        total_records += len(chunk_df)

    if total_records == 0:
        pd.DataFrame(columns=historical_output_columns).to_csv(historical_clean_output_path, index=False)

    print(f"Archivos internos inspeccionados: {len(inventory_records):,}")
    print(f"Archivos JSON encontrados: {json_files_seen:,}")
    print(f"Listas JSON con estructura de estación detectadas: {processed_json_files:,}")
    print(f"Registros históricos escritos: {total_records:,}")
    print(f"Inventario debug generado: {historical_zip_inventory_debug_path.relative_to(config.project_root)}")
    print(f"Archivo generado: {historical_clean_output_path.relative_to(config.project_root)}")

    station_status_history_2022_clean_df = pd.read_csv(
        historical_clean_output_path,
        parse_dates=["snapshot_datetime"],
        low_memory=False,
    )

    numeric_columns = [
        "station_id_historical",
        "station_number",
        "station_latitude",
        "station_longitude",
        "station_capacity",
        "num_docks_available",
        "num_bikes_available",
        "reservations_count",
        "station_light_status",
        "is_active",
        "is_not_available",
        "occupation_ratio",
    ]

    for column in numeric_columns:
        if column in station_status_history_2022_clean_df.columns:
            station_status_history_2022_clean_df[column] = pd.to_numeric(
                station_status_history_2022_clean_df[column],
                errors="coerce",
            )

    print("Validación rápida del histórico limpio:")
    print(f"Filas: {len(station_status_history_2022_clean_df):,}")
    print(f"Columnas: {len(station_status_history_2022_clean_df.columns)}")

    if not station_status_history_2022_clean_df.empty:
        print(f"Fecha mínima: {station_status_history_2022_clean_df['snapshot_datetime'].min()}")
        print(f"Fecha máxima: {station_status_history_2022_clean_df['snapshot_datetime'].max()}")
        print(f"Estaciones únicas: {station_status_history_2022_clean_df['station_id_historical'].nunique()}")
        display(station_status_history_2022_clean_df.head())
    else:
        print(
            "No se han extraído registros de estaciones. "
            "Revisa el inventario debug para ver la estructura interna del ZIP."
        )
        display(inventory_df.head(30))


Procesando ZIP histórico. Esta celda puede tardar varios minutos.
Registros escritos: 195,360
Registros escritos: 382,536
Registros escritos: 576,576
Registros escritos: 765,864
Registros escritos: 960,960
Registros escritos: 1,157,112
Registros escritos: 1,346,400
Registros escritos: 1,522,752
Registros escritos: 1,729,992
Registros escritos: 1,921,392
Registros escritos: 2,117,016
Registros escritos: 2,306,832
Archivos internos inspeccionados: 36
Archivos JSON encontrados: 12
Listas JSON con estructura de estación detectadas: 8,738
Registros históricos escritos: 2,306,832
Inventario debug generado: data\interim\bicimad\historical_zip_inventory_debug.csv
Archivo generado: data\interim\bicimad\station_status_history_2022_clean.csv
Validación rápida del histórico limpio:
Filas: 2,306,832
Columnas: 16
Fecha mínima: 2022-01-01 00:00:00
Fecha máxima: 2022-12-01 00:00:00
Estaciones únicas: 264


,source_month,snapshot_datetime,station_id_historical,station_number,station_name,station_address,station_latitude,station_longitude,station_capacity,num_docks_available,num_bikes_available,reservations_count,station_light_status,is_active,is_not_available,occupation_ratio
0,2022-01,2022-01-01,1,NaN,Puerta del Sol A,Puerta del Sol nº 1,40.417214,-3.701834,30,0,0,0,3.0,1,1,0.000000
1,2022-01,2022-01-01,2,NaN,Puerta del Sol B,Puerta del Sol nº 1,40.417313,-3.701603,30,0,0,0,3.0,1,1,0.000000
2,2022-01,2022-01-01,3,2.0,Miguel Moya,Calle Miguel Moya nº 1,40.420589,-3.705842,24,16,7,0,0.0,1,0,0.291667
3,2022-01,2022-01-01,4,3.0,Plaza Conde Suchil,Plaza del Conde del Valle de Súchil nº 3,40.430294,-3.706917,18,1,14,0,1.0,1,0,0.777778
4,2022-01,2022-01-01,5,4.0,Malasaña,Calle Manuela Malasaña nº 5,40.428552,-3.702587,24,2,17,0,1.0,1,0,0.708333


## 12. Creación de la base enriquecida intermedia

Esta sección une el histórico limpio de BiciMAD 2022 con calendario laboral y meteorología diaria.

El resultado es:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`


In [17]:
enriched_base_output_path = config.interim_bicimad_dir / "station_status_history_2022_enriched_base.csv"
processed_modeling_base_output_path = config.processed_dir / "station_status_history_2022_modeling_base.csv"


def parse_datetime_safely(series: pd.Series) -> pd.Series:
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")


if historical_clean_output_path.exists():
    station_history_df = pd.read_csv(
        historical_clean_output_path,
        low_memory=False,
    )
else:
    station_history_df = pd.DataFrame()

calendar_clean_path = config.interim_calendar_dir / "madrid_working_calendar_clean.csv"
weather_join_ready_path = config.interim_weather_dir / "aemet_daily_weather_madrid_2022_join_ready.csv"

calendar_for_join_df = (
    pd.read_csv(calendar_clean_path)
    if calendar_clean_path.exists()
    else pd.DataFrame()
)

weather_for_join_df = (
    pd.read_csv(weather_join_ready_path)
    if weather_join_ready_path.exists()
    else pd.DataFrame()
)

if station_history_df.empty:
    enriched_base_df = pd.DataFrame()
    print("No se puede crear la base enriquecida porque falta el histórico limpio.")
else:
    enriched_base_df = station_history_df.copy()

    enriched_base_df["snapshot_datetime"] = parse_datetime_safely(
        enriched_base_df["snapshot_datetime"]
    )

    missing_snapshot_datetime = int(enriched_base_df["snapshot_datetime"].isna().sum())

    if missing_snapshot_datetime > 0:
        print(
            "Aviso: hay registros sin snapshot_datetime válido. "
            f"Registros afectados: {missing_snapshot_datetime:,}"
        )

    enriched_base_df["date"] = enriched_base_df["snapshot_datetime"].dt.normalize()
    enriched_base_df["snapshot_hour"] = enriched_base_df["snapshot_datetime"].dt.hour
    enriched_base_df["snapshot_day_of_week"] = enriched_base_df["snapshot_datetime"].dt.dayofweek
    enriched_base_df["snapshot_month"] = enriched_base_df["snapshot_datetime"].dt.month

    if not calendar_for_join_df.empty:
        calendar_for_join_df["date"] = parse_datetime_safely(calendar_for_join_df["date"]).dt.normalize()

        calendar_cols = [
            "date",
            "year",
            "month",
            "day_of_week",
            "day_of_week_name_es",
            "day_type",
            "is_working_day",
            "is_non_working_day",
            "is_weekend",
            "is_holiday",
            "holiday_type",
            "holiday_name",
        ]

        available_calendar_cols = [
            col for col in calendar_cols
            if col in calendar_for_join_df.columns
        ]

        enriched_base_df = enriched_base_df.merge(
            calendar_for_join_df[available_calendar_cols],
            on="date",
            how="left",
        )
    else:
        print("Calendario limpio no disponible. Se crearán variables temporales básicas.")

        enriched_base_df["day_type"] = np.select(
            [
                enriched_base_df["snapshot_day_of_week"] == 5,
                enriched_base_df["snapshot_day_of_week"] == 6,
            ],
            ["sabado", "domingo"],
            default="laborable",
        )

        enriched_base_df["is_weekend"] = enriched_base_df["snapshot_day_of_week"].isin([5, 6])
        enriched_base_df["is_working_day"] = ~enriched_base_df["is_weekend"]
        enriched_base_df["is_non_working_day"] = enriched_base_df["is_weekend"]
        enriched_base_df["is_holiday"] = False
        enriched_base_df["holiday_type"] = pd.NA
        enriched_base_df["holiday_name"] = pd.NA

    if not weather_for_join_df.empty:
        weather_for_join_df["date"] = parse_datetime_safely(weather_for_join_df["date"]).dt.normalize()

        enriched_base_df = enriched_base_df.merge(
            weather_for_join_df,
            on="date",
            how="left",
        )
    else:
        print("Meteorología limpia no disponible. La base se generará sin variables meteorológicas.")

    enriched_base_output_path.parent.mkdir(parents=True, exist_ok=True)
    processed_modeling_base_output_path.parent.mkdir(parents=True, exist_ok=True)

    enriched_base_df.to_csv(enriched_base_output_path, index=False)
    enriched_base_df.to_csv(processed_modeling_base_output_path, index=False)

    print(f"Archivo intermedio generado: {enriched_base_output_path.relative_to(config.project_root)}")
    print(f"Copia para modelado generada: {processed_modeling_base_output_path.relative_to(config.project_root)}")
    print(f"Filas: {len(enriched_base_df):,}")
    print(f"Columnas: {len(enriched_base_df.columns)}")
    print(f"snapshot_datetime nulos: {int(enriched_base_df['snapshot_datetime'].isna().sum()):,}")

    if "weather_temperature_mean_c" in enriched_base_df.columns:
        print(f"weather_temperature_mean_c nulos: {int(enriched_base_df['weather_temperature_mean_c'].isna().sum()):,}")

    display(enriched_base_df.head())


Archivo intermedio generado: data\interim\bicimad\station_status_history_2022_enriched_base.csv
Copia para modelado generada: data\processed\station_status_history_2022_modeling_base.csv
Filas: 2,306,832
Columnas: 41
snapshot_datetime nulos: 0
weather_temperature_mean_c nulos: 0


,source_month,snapshot_datetime,station_id_historical,station_number,station_name,station_address,station_latitude,station_longitude,station_capacity,num_docks_available,...,weather_temperature_mean_c,weather_temperature_min_c,weather_temperature_max_c,weather_precipitation_mm,weather_humidity_mean_pct,weather_station_count,weather_temperature_available_count,weather_precipitation_available_count,weather_humidity_available_count,weather_has_precipitation
0,2022-01,2022-01-01,1,1a,Puerta del Sol A,Puerta del Sol nº 1,40.417214,-3.701834,30,0,...,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
1,2022-01,2022-01-01,2,1b,Puerta del Sol B,Puerta del Sol nº 1,40.417313,-3.701603,30,0,...,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
2,2022-01,2022-01-01,3,2,Miguel Moya,Calle Miguel Moya nº 1,40.420589,-3.705842,24,16,...,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
3,2022-01,2022-01-01,4,3,Plaza Conde Suchil,Plaza del Conde del Valle de Súchil nº 3,40.430294,-3.706917,18,1,...,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False
4,2022-01,2022-01-01,5,4,Malasaña,Calle Manuela Malasaña nº 5,40.428552,-3.702587,24,2,...,10.06,2.36,17.76,0.0,63.6,5,5,5,5,False


## 13. Validación de la base enriquecida

Esta sección comprueba que la base enriquecida tiene coherencia mínima antes de pasar al contrato de datos y al EDA.


In [18]:
if enriched_base_output_path.exists():
    enriched_validation_df = pd.read_csv(
        enriched_base_output_path,
        parse_dates=["snapshot_datetime", "date"],
        low_memory=False,
    )
else:
    enriched_validation_df = pd.DataFrame()

if enriched_validation_df.empty:
    print("No existe base enriquecida para validar.")
else:
    validation_summary = pd.DataFrame(
        [
            {"Métrica": "Filas", "Valor": len(enriched_validation_df)},
            {"Métrica": "Columnas", "Valor": len(enriched_validation_df.columns)},
            {"Métrica": "Fecha mínima snapshot", "Valor": enriched_validation_df["snapshot_datetime"].min()},
            {"Métrica": "Fecha máxima snapshot", "Valor": enriched_validation_df["snapshot_datetime"].max()},
            {"Métrica": "Estaciones históricas únicas", "Valor": enriched_validation_df["station_id_historical"].nunique()},
            {"Métrica": "Nulos en snapshot_datetime", "Valor": int(enriched_validation_df["snapshot_datetime"].isna().sum())},
            {"Métrica": "Nulos en station_id_historical", "Valor": int(enriched_validation_df["station_id_historical"].isna().sum())},
            {"Métrica": "Nulos en num_bikes_available", "Valor": int(enriched_validation_df["num_bikes_available"].isna().sum())},
            {"Métrica": "Nulos en num_docks_available", "Valor": int(enriched_validation_df["num_docks_available"].isna().sum())},
        ]
    )
    display(validation_summary)

    if "weather_temperature_mean_c" in enriched_validation_df.columns:
        print("Variables meteorológicas disponibles en la base enriquecida.")
    else:
        print("Variables meteorológicas no disponibles en la base enriquecida.")


,Métrica,Valor
0,Filas,2306832
1,Columnas,41
2,Fecha mínima snapshot,2022-01-01 00:00:00
3,Fecha máxima snapshot,2022-12-01 00:00:00
4,Estaciones históricas únicas,264
5,Nulos en snapshot_datetime,0
6,Nulos en station_id_historical,0
7,Nulos en num_bikes_available,0
8,Nulos en num_docks_available,0


Variables meteorológicas disponibles en la base enriquecida.


## 14. Contrato de datos para modelado

Esta sección documenta qué archivo debe usar el siguiente notebook y qué columnas principales contiene.

El archivo recomendado para preprocesamiento y modelado es:

`data/processed/station_status_history_2022_modeling_base.csv`

La versión equivalente dentro del flujo intermedio se conserva en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

Esta base no es todavía el dataset final de Machine Learning. Todavía falta crear el target definitivo, decidir variables predictoras, dividir los datos temporalmente y aplicar el preprocesamiento necesario.


In [19]:
if enriched_validation_df.empty:
    print("No se puede generar contrato de datos porque la base enriquecida no existe todavía.")
else:
    contract_rows = []

    for column in enriched_validation_df.columns:
        contract_rows.append(
            {
                "Columna": column,
                "Tipo pandas": str(enriched_validation_df[column].dtype),
                "Nulos": int(enriched_validation_df[column].isna().sum()),
                "Porcentaje nulos": round(enriched_validation_df[column].isna().mean() * 100, 2),
                "Ejemplo": enriched_validation_df[column].dropna().iloc[0] if enriched_validation_df[column].dropna().shape[0] else pd.NA,
            }
        )

    data_contract_df = pd.DataFrame(contract_rows)
    display(data_contract_df)

    data_contract_path = config.interim_bicimad_dir / "station_status_history_2022_enriched_base_contract.csv"
    data_contract_df.to_csv(data_contract_path, index=False)
    print(f"Contrato de datos generado: {data_contract_path.relative_to(config.project_root)}")


,Columna,Tipo pandas,Nulos,Porcentaje nulos,Ejemplo
0,source_month,str,0,0.00,2022-01
1,snapshot_datetime,datetime64[us],0,0.00,2022-01-01 00:00:00
2,station_id_historical,int64,0,0.00,1
3,station_number,str,0,0.00,1a
4,station_name,str,0,0.00,Puerta del Sol A
5,station_address,str,0,0.00,Puerta del Sol nº 1
6,station_latitude,float64,0,0.00,40.417214
7,station_longitude,float64,0,0.00,-3.701834
8,station_capacity,int64,0,0.00,30
9,num_docks_available,int64,0,0.00,0


Contrato de datos generado: data\interim\bicimad\station_status_history_2022_enriched_base_contract.csv


## 15. Reglas de limpieza para EDA

El EDA necesita distinguir entre registros normales y registros donde la estación no está disponible o parece operativamente restringida.

Reglas usadas:

- `available_record`: la estación no está marcada como no disponible.
- `capacity_balance`: capacidad menos bicicletas disponibles menos anclajes disponibles.
- `operationally_constrained`: estación disponible, pero con un balance anómalo alto.
- `normal_operational_record`: registro disponible y no restringido.
- `near_empty`: estación casi vacía.
- `near_full`: estación casi llena.


In [20]:
if enriched_validation_df.empty:
    eda_df = pd.DataFrame()
    print("No hay base enriquecida para preparar EDA.")
else:
    eda_df = enriched_validation_df.copy()

    for col in ["station_capacity", "num_bikes_available", "num_docks_available", "is_not_available"]:
        if col in eda_df.columns:
            eda_df[col] = pd.to_numeric(eda_df[col], errors="coerce")

    eda_df["available_record"] = eda_df["is_not_available"].fillna(0).eq(0)
    eda_df["capacity_balance"] = (
        eda_df["station_capacity"]
        - eda_df["num_bikes_available"]
        - eda_df["num_docks_available"]
    )

    eda_df["operationally_constrained"] = (
        eda_df["available_record"]
        & (eda_df["station_capacity"].notna())
        & (eda_df["capacity_balance"] >= 0.5 * eda_df["station_capacity"])
    )

    eda_df["normal_operational_record"] = (
        eda_df["available_record"] & ~eda_df["operationally_constrained"]
    )

    eda_df["empty"] = eda_df["normal_operational_record"] & eda_df["num_bikes_available"].eq(0)
    eda_df["near_empty"] = eda_df["normal_operational_record"] & eda_df["num_bikes_available"].le(2)
    eda_df["full"] = eda_df["normal_operational_record"] & eda_df["num_docks_available"].eq(0)
    eda_df["near_full"] = eda_df["normal_operational_record"] & eda_df["num_docks_available"].le(2)

    print("Reglas de EDA aplicadas correctamente.")
    print(f"Filas para EDA: {len(eda_df):,}")


Reglas de EDA aplicadas correctamente.
Filas para EDA: 2,306,832


## 16. EDA global

Visión general del estado operativo de la red.


In [21]:
if eda_df.empty:
    print("No hay datos para EDA global.")
else:
    total_records = len(eda_df)
    available_records = int(eda_df["available_record"].sum())
    normal_records = int(eda_df["normal_operational_record"].sum())

    global_summary = pd.DataFrame(
        [
            {"Métrica": "Registros totales", "Valor": total_records, "Porcentaje": 100.0},
            {"Métrica": "Registros disponibles", "Valor": available_records, "Porcentaje": round(available_records / total_records * 100, 2)},
            {"Métrica": "Registros no disponibles", "Valor": total_records - available_records, "Porcentaje": round((total_records - available_records) / total_records * 100, 2)},
            {"Métrica": "Registros normales para EDA", "Valor": normal_records, "Porcentaje": round(normal_records / total_records * 100, 2)},
            {"Métrica": "Estaciones casi vacías", "Valor": int(eda_df["near_empty"].sum()), "Porcentaje": round(eda_df["near_empty"].mean() * 100, 2)},
            {"Métrica": "Estaciones casi llenas", "Valor": int(eda_df["near_full"].sum()), "Porcentaje": round(eda_df["near_full"].mean() * 100, 2)},
        ]
    )

    display(global_summary)


,Métrica,Valor,Porcentaje
0,Registros totales,2306832,100.00
1,Registros disponibles,2233556,96.82
2,Registros no disponibles,73276,3.18
3,Registros normales para EDA,2230403,96.69
4,Estaciones casi vacías,220444,9.56
5,Estaciones casi llenas,128333,5.56


## 17. EDA por estación

Análisis de estaciones con mayor riesgo de quedarse casi vacías o casi llenas.


In [22]:
if eda_df.empty:
    print("No hay datos para EDA por estación.")
else:
    station_eda_df = (
        eda_df
        .groupby(["station_id_historical", "station_name"], dropna=False)
        .agg(
            records=("station_id_historical", "count"),
            normal_operational_records=("normal_operational_record", "sum"),
            not_available_pct=("available_record", lambda s: round((1 - s.mean()) * 100, 2)),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            empty_pct=("empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            full_pct=("full", lambda s: round(s.mean() * 100, 2)),
            mean_bikes_available=("num_bikes_available", "mean"),
            mean_docks_available=("num_docks_available", "mean"),
            median_capacity=("station_capacity", "median"),
        )
        .reset_index()
    )

    station_eda_df["is_reliable_for_operational_risk"] = (
        (station_eda_df["normal_operational_records"] >= 1000)
        & (station_eda_df["not_available_pct"] <= 20)
    )

    reliable_station_eda_df = station_eda_df[station_eda_df["is_reliable_for_operational_risk"]].copy()

    print("Top estaciones con mayor riesgo de estar casi vacías:")
    display(
        reliable_station_eda_df
        .sort_values("near_empty_pct", ascending=False)
        .head(15)
    )

    print("Top estaciones con mayor riesgo de estar casi llenas:")
    display(
        reliable_station_eda_df
        .sort_values("near_full_pct", ascending=False)
        .head(15)
    )


Top estaciones con mayor riesgo de estar casi vacías:


,station_id_historical,station_name,records,normal_operational_records,not_available_pct,near_empty_pct,empty_pct,near_full_pct,full_pct,mean_bikes_available,mean_docks_available,median_capacity,is_reliable_for_operational_risk
159,165,Paseo de la Castellana - Glorieta de Emilio Ca...,8738,8699,0.39,54.83,21.24,9.76,3.89,3.316091,7.973907,12.0,True
10,11,Marqués de la Ensenada,8738,8738,0.00,36.67,13.07,1.60,0.55,5.779240,16.433394,24.0,True
263,270,Zurbano,8738,8735,0.00,32.80,8.50,0.41,0.09,5.250629,16.820897,24.0,True
92,98,Plaza de Colón,8738,8364,0.71,32.42,11.34,0.37,0.15,5.005493,16.339666,24.0,True
106,112,Colón B,8738,8708,0.19,31.53,12.15,1.54,0.48,5.148432,12.279927,18.0,True
259,266,Ciudad Universitaria 1,8738,8562,2.00,30.65,11.26,0.26,0.06,5.789769,17.429046,24.0,True
93,99,Biblioteca Nacional,8738,8685,0.56,30.54,8.77,5.47,1.79,6.273976,13.187343,21.0,True
85,91,Cibeles,8738,8625,1.24,28.59,8.98,6.36,2.25,6.850423,12.693980,21.0,True
94,100,Villanueva,8738,8393,3.94,28.15,8.55,2.84,1.17,6.637102,15.457313,24.0,True
107,113,Columela,8738,8682,0.21,25.92,8.80,4.21,1.57,7.697757,15.281872,24.0,True


Top estaciones con mayor riesgo de estar casi llenas:


,station_id_historical,station_name,records,normal_operational_records,not_available_pct,near_empty_pct,empty_pct,near_full_pct,full_pct,mean_bikes_available,mean_docks_available,median_capacity,is_reliable_for_operational_risk
44,48,Plaza de Nelson Mandela,8738,8544,1.91,6.97,1.37,25.66,7.50,10.978141,6.827077,21.0,True
38,42,Plaza de los Carros,8738,8559,1.87,6.23,1.13,24.74,11.41,13.718471,8.302815,24.0,True
32,36,Plaza de la Provincia,8738,8474,2.98,2.41,0.47,22.75,6.59,10.867132,5.773976,18.0,True
40,44,Conde de Romanones,8738,8734,0.00,3.82,0.63,21.76,7.23,14.318379,7.903983,24.0,True
41,45,Antón Martín,8738,8466,1.80,10.06,1.75,20.74,6.81,8.562600,6.613985,18.0,True
214,220,Marqués de Vadillo,8738,8717,0.00,3.79,0.64,20.58,8.87,13.629091,8.274891,24.0,True
39,43,Plaza de la Cebada,8738,8556,2.00,5.39,0.94,20.14,7.62,14.680247,9.301099,27.0,True
37,41,Plaza de San Francisco,8738,8605,1.26,6.10,1.05,19.84,8.32,12.981689,8.993477,24.0,True
52,56,Plaza de Santa Ana,8738,8708,0.18,1.77,0.19,18.95,5.93,13.996796,7.780041,24.0,True
181,187,Puente de Vallecas,8738,8738,0.00,2.05,0.25,18.28,7.02,14.295148,7.939460,24.0,True


## 18. EDA temporal

Análisis por hora, mes y tipo de día.


In [23]:
if eda_df.empty:
    print("No hay datos para EDA temporal.")
else:
    by_hour_df = (
        eda_df
        .groupby("snapshot_hour", dropna=False)
        .agg(
            records=("snapshot_hour", "count"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            mean_bikes_available=("num_bikes_available", "mean"),
            mean_docks_available=("num_docks_available", "mean"),
        )
        .reset_index()
    )

    print("Riesgo por hora:")
    display(by_hour_df)

    by_month_df = (
        eda_df
        .groupby("snapshot_month", dropna=False)
        .agg(
            records=("snapshot_month", "count"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
        )
        .reset_index()
    )

    print("Riesgo por mes:")
    display(by_month_df)

    if "day_type" in eda_df.columns:
        by_day_type_df = (
            eda_df
            .groupby("day_type", dropna=False)
            .agg(
                records=("day_type", "count"),
                near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
                near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            )
            .reset_index()
        )

        print("Riesgo por tipo de día:")
        display(by_day_type_df)


Riesgo por hora:


,snapshot_hour,records,near_empty_pct,near_full_pct,mean_bikes_available,mean_docks_available
0,0,2306832,9.56,5.56,9.769188,12.108733


Riesgo por mes:


,snapshot_month,records,near_empty_pct,near_full_pct
0,1,195360,6.80,6.16
1,2,176352,7.24,5.81
2,3,207240,7.42,5.50
3,4,189816,8.21,5.55
4,5,195624,11.58,7.07
5,6,187176,12.40,7.37
6,7,194040,10.71,6.31
7,8,196152,6.96,3.66
8,9,189288,15.25,7.08
9,10,195096,13.41,5.30


Riesgo por tipo de día:


,day_type,records,near_empty_pct,near_full_pct
0,domingo,195624,11.58,7.07
1,festivo,384648,7.92,5.28
2,laborable,1531464,9.22,5.48
3,sabado,195096,13.41,5.30


## 19. EDA meteorológica

Análisis sencillo de disponibilidad según precipitación y temperatura.


In [24]:
if eda_df.empty:
    print("No hay datos para EDA meteorológica.")
elif "weather_temperature_mean_c" not in eda_df.columns:
    print("La base no contiene variables meteorológicas.")
else:
    weather_eda_df = eda_df.copy()

    if "weather_has_precipitation" in weather_eda_df.columns:
        by_precip_df = (
            weather_eda_df
            .groupby("weather_has_precipitation", dropna=False)
            .agg(
                records=("weather_has_precipitation", "count"),
                near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
                near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            )
            .reset_index()
        )

        print("Riesgo según precipitación diaria:")
        display(by_precip_df)

    weather_eda_df["temperature_bucket"] = pd.cut(
        weather_eda_df["weather_temperature_mean_c"],
        bins=[-20, 5, 10, 15, 20, 25, 30, 50],
        labels=["<=5", "5-10", "10-15", "15-20", "20-25", "25-30", ">30"],
    )

    by_temp_df = (
        weather_eda_df
        .groupby("temperature_bucket", observed=False)
        .agg(
            records=("temperature_bucket", "count"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
        )
        .reset_index()
    )

    print("Riesgo según temperatura media diaria:")
    display(by_temp_df)


Riesgo según precipitación diaria:


,weather_has_precipitation,records,near_empty_pct,near_full_pct
0,False,1921392,9.48,5.41
1,True,385440,9.92,6.32


Riesgo según temperatura media diaria:


,temperature_bucket,records,near_empty_pct,near_full_pct
0,<=5,0,NaN,NaN
1,5-10,381216,6.96,4.08
2,10-15,578952,7.15,5.82
3,15-20,580008,11.38,5.59
4,20-25,570504,12.77,6.91
5,25-30,0,NaN,NaN
6,>30,196152,6.96,3.66


## 20. EDA estación + hora

Análisis combinado para detectar estaciones y franjas horarias especialmente críticas.


In [25]:
if eda_df.empty:
    print("No hay datos para EDA estación + hora.")
else:
    station_hour_df = (
        eda_df
        .groupby(["station_id_historical", "station_name", "snapshot_hour"], dropna=False)
        .agg(
            records=("station_id_historical", "count"),
            normal_operational_records=("normal_operational_record", "sum"),
            near_empty_pct=("near_empty", lambda s: round(s.mean() * 100, 2)),
            empty_pct=("empty", lambda s: round(s.mean() * 100, 2)),
            near_full_pct=("near_full", lambda s: round(s.mean() * 100, 2)),
            full_pct=("full", lambda s: round(s.mean() * 100, 2)),
        )
        .reset_index()
    )

    station_hour_reliable_df = station_hour_df[station_hour_df["normal_operational_records"] >= 100].copy()

    print("Top estación + hora con mayor riesgo de estar casi vacía:")
    display(
        station_hour_reliable_df
        .sort_values("near_empty_pct", ascending=False)
        .head(20)
    )

    print("Top estación + hora con mayor riesgo de estar casi llena:")
    display(
        station_hour_reliable_df
        .sort_values("near_full_pct", ascending=False)
        .head(20)
    )


Top estación + hora con mayor riesgo de estar casi vacía:


,station_id_historical,station_name,snapshot_hour,records,normal_operational_records,near_empty_pct,empty_pct,near_full_pct,full_pct
159,165,Paseo de la Castellana - Glorieta de Emilio Ca...,0,8738,8699,54.83,21.24,9.76,3.89
10,11,Marqués de la Ensenada,0,8738,8738,36.67,13.07,1.60,0.55
263,270,Zurbano,0,8738,8735,32.80,8.50,0.41,0.09
92,98,Plaza de Colón,0,8738,8364,32.42,11.34,0.37,0.15
106,112,Colón B,0,8738,8708,31.53,12.15,1.54,0.48
259,266,Ciudad Universitaria 1,0,8738,8562,30.65,11.26,0.26,0.06
93,99,Biblioteca Nacional,0,8738,8685,30.54,8.77,5.47,1.79
85,91,Cibeles,0,8738,8625,28.59,8.98,6.36,2.25
94,100,Villanueva,0,8738,8393,28.15,8.55,2.84,1.17
107,113,Columela,0,8738,8682,25.92,8.80,4.21,1.57


Top estación + hora con mayor riesgo de estar casi llena:


,station_id_historical,station_name,snapshot_hour,records,normal_operational_records,near_empty_pct,empty_pct,near_full_pct,full_pct
44,48,Plaza de Nelson Mandela,0,8738,8544,6.97,1.37,25.66,7.50
38,42,Plaza de los Carros,0,8738,8559,6.23,1.13,24.74,11.41
32,36,Plaza de la Provincia,0,8738,8474,2.41,0.47,22.75,6.59
40,44,Conde de Romanones,0,8738,8734,3.82,0.63,21.76,7.23
41,45,Antón Martín,0,8738,8466,10.06,1.75,20.74,6.81
214,220,Marqués de Vadillo,0,8738,8717,3.79,0.64,20.58,8.87
39,43,Plaza de la Cebada,0,8738,8556,5.39,0.94,20.14,7.62
37,41,Plaza de San Francisco,0,8738,8605,6.10,1.05,19.84,8.32
52,56,Plaza de Santa Ana,0,8738,8708,1.77,0.19,18.95,5.93
181,187,Puente de Vallecas,0,8738,8738,2.05,0.25,18.28,7.02


## 21. Hallazgos principales

Esta sección resume los hallazgos principales detectados por el EDA.

El contenido se genera a partir de las tablas anteriores. Si alguna fuente no se ha podido descargar o limpiar, los hallazgos deberán interpretarse con cautela.


In [26]:
findings = []

if not eda_df.empty:
    near_empty_pct = round(eda_df["near_empty"].mean() * 100, 2)
    near_full_pct = round(eda_df["near_full"].mean() * 100, 2)

    findings.append(f"El porcentaje global de registros casi vacíos es aproximadamente {near_empty_pct}%.")
    findings.append(f"El porcentaje global de registros casi llenos es aproximadamente {near_full_pct}%.")

    if "by_hour_df" in globals() and not by_hour_df.empty:
        by_hour_valid_df = by_hour_df.dropna(subset=["snapshot_hour"]).copy()

        if not by_hour_valid_df.empty:
            top_empty_hour = by_hour_valid_df.sort_values("near_empty_pct", ascending=False).iloc[0]

            findings.append(
                f"La hora con mayor riesgo global de estación casi vacía es {int(top_empty_hour['snapshot_hour'])}:00 "
                f"con {top_empty_hour['near_empty_pct']}%."
            )
        else:
            findings.append(
                "No se ha podido identificar una hora crítica porque snapshot_hour no contiene valores válidos."
            )

    if "reliable_station_eda_df" in globals() and not reliable_station_eda_df.empty:
        top_empty_station = reliable_station_eda_df.sort_values("near_empty_pct", ascending=False).iloc[0]
        top_full_station = reliable_station_eda_df.sort_values("near_full_pct", ascending=False).iloc[0]

        findings.append(
            f"La estación con mayor riesgo de estar casi vacía es {top_empty_station['station_name']} "
            f"con {top_empty_station['near_empty_pct']}%."
        )

        findings.append(
            f"La estación con mayor riesgo de estar casi llena es {top_full_station['station_name']} "
            f"con {top_full_station['near_full_pct']}%."
        )

if not findings:
    findings.append("No se han podido generar hallazgos porque no hay datos suficientes.")

for index, finding in enumerate(findings, start=1):
    print(f"{index}. {finding}")


1. El porcentaje global de registros casi vacíos es aproximadamente 9.56%.
2. El porcentaje global de registros casi llenos es aproximadamente 5.56%.
3. La hora con mayor riesgo global de estación casi vacía es 0:00 con 9.56%.
4. La estación con mayor riesgo de estar casi vacía es Paseo de la Castellana - Glorieta de Emilio Castelar con 54.83%.
5. La estación con mayor riesgo de estar casi llena es Plaza de Nelson Mandela con 25.66%.


## 22. Limitaciones

Principales limitaciones de esta fase:

- El histórico de estaciones corresponde a 2022.
- La red actual de BiciMAD puede no coincidir exactamente con la red histórica.
- Las fuentes GBFS actuales son útiles para referencia o demo, pero no sustituyen al histórico.
- La meteorología se integra a nivel diario, no horario.
- Este notebook no define el target final ni entrena modelos.
- La fase de modelado deberá evitar fuga de información y respetar el orden temporal.


## 23. Entrega para el siguiente notebook

El archivo recomendado para continuar el proyecto es:

**Opción A (base completa):** `data/processed/station_status_history_2022_modeling_base.csv`
- 31 columnas, incluye todas las variables originales, calendario y meteorología.

**Opción B (base limpia):** `data/processed/station_status_history_2022_modeling_base_clean.csv`
- 21 columnas, elimina redundancias y columnas sin valor predictivo. Recomendada para preprocesamiento.

La versión trazable equivalente queda en:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

In [27]:
handoff_summary = pd.DataFrame(
    [
        {
            "Elemento": "Archivo recomendado para modelado",
            "Valor": "data/processed/station_status_history_2022_modeling_base.csv",
        },
        {
            "Elemento": "Archivo intermedio trazable",
            "Valor": "data/interim/bicimad/station_status_history_2022_enriched_base.csv",
        },
        {
            "Elemento": "Tipo de archivo",
            "Valor": "Base histórica limpia y enriquecida",
        },
        {
            "Elemento": "Uso recomendado",
            "Valor": "Entrada para preprocesamiento y modelado",
        },
        {
            "Elemento": "Granularidad",
            "Valor": "Una fila por estación y momento temporal",
        },
        {
            "Elemento": "Identificador principal",
            "Valor": "station_id_historical",
        },
        {
            "Elemento": "Fecha/hora principal",
            "Valor": "snapshot_datetime",
        },
        {
            "Elemento": "No contiene",
            "Valor": "Target final, train/test split, escalado, encoding ni modelo",
        },
        {
            "Elemento": "Notebook siguiente",
            "Valor": "notebooks/02_preprocesamiento.ipynb",
        },
    ]
)

display(handoff_summary)


,Elemento,Valor
0,Archivo recomendado para modelado,data/processed/station_status_history_2022_mod...
1,Archivo intermedio trazable,data/interim/bicimad/station_status_history_20...
2,Tipo de archivo,Base histórica limpia y enriquecida
3,Uso recomendado,Entrada para preprocesamiento y modelado
4,Granularidad,Una fila por estación y momento temporal
5,Identificador principal,station_id_historical
6,Fecha/hora principal,snapshot_datetime
7,No contiene,"Target final, train/test split, escalado, enco..."
8,Notebook siguiente,notebooks/02_preprocesamiento.ipynb


## 24. Archivos generados

Esta sección lista los archivos principales generados por el notebook.


In [28]:
generated_files = [
    config.raw_bicimad_dir / "gbfs_station_information_raw.json",
    config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json",
    config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv",
    config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip",
    config.raw_calendar_dir / "madrid_working_calendar_raw.csv",
    config.raw_weather_dir / "aemet_stations_inventory_raw.json",
    config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json",
    config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
    config.interim_bicimad_dir / "stations_clean.csv",
    config.interim_bicimad_dir / "station_status_snapshot_clean.csv",
    config.interim_bicimad_dir / "bicimad_daily_trips_clean.csv",
    config.interim_calendar_dir / "madrid_working_calendar_clean.csv",
    config.interim_weather_dir / "aemet_stations_inventory_clean.csv",
    config.interim_weather_dir / "aemet_daily_climate_selected_stations_2022_clean.csv",
    config.interim_weather_dir / "aemet_daily_weather_madrid_2022_join_ready.csv",
    config.interim_weather_dir / "aemet_forecast_hourly_madrid_clean.csv",
    config.interim_bicimad_dir / "station_status_history_2022_clean.csv",
    config.interim_bicimad_dir / "station_status_history_2022_enriched_base.csv",
    config.processed_dir / "station_status_history_2022_modeling_base.csv",
    config.processed_dir / "station_status_history_2022_modeling_base_clean.csv", 
    config.interim_bicimad_dir / "station_status_history_2022_enriched_base_contract.csv",
]

generated_files_df = pd.DataFrame(
    [
        {
            "Archivo": str(path.relative_to(config.project_root)),
            "Existe": path.exists(),
            "Tamaño MB": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0,
        }
        for path in generated_files
    ]
)

display(generated_files_df)

print("Notebook finalizado.")
print("Recuerda: los datos están protegidos por .gitignore y no se suben al repositorio.")


,Archivo,Existe,Tamaño MB
0,data\raw\bicimad\gbfs_station_information_raw....,True,0.51
1,data\raw\bicimad\gbfs_station_status_snapshot_...,True,0.27
2,data\raw\bicimad\bicimad_daily_trips_raw.csv,True,0.01
3,data\raw\bicimad\bicimad_station_status_histor...,True,300.74
4,data\raw\calendar\madrid_working_calendar_raw.csv,True,0.16
5,data\raw\weather\aemet_stations_inventory_raw....,True,0.18
6,data\raw\weather\aemet_daily_climate_selected_...,True,1.17
7,data\raw\weather\aemet_forecast_hourly_madrid_...,True,0.04
8,data\interim\bicimad\stations_clean.csv,True,0.06
9,data\interim\bicimad\station_status_snapshot_c...,True,0.04


Notebook finalizado.
Recuerda: los datos están protegidos por .gitignore y no se suben al repositorio.


## 25. Generación de dataset limpio para modelado

Esta sección elimina columnas redundantes, de metadata o sin valor predictivo para facilitar el trabajo del notebook de preprocesamiento y modelado.

Columnas eliminadas:
- `station_number`: redundante con `station_id_historical`.
- `station_light_status`: código operativo interno sin significado documentado.
- `source_month`: redundante con `snapshot_month`.
- `date`: redundante con `snapshot_datetime`.
- `month`: duplicado de `snapshot_month`.
- `day_of_week`: duplicado de `snapshot_day_of_week`.
- `is_non_working_day`: duplicado de `~is_working_day`.
- `is_holiday`: duplicado de `day_type == "festivo"`.
- `holiday_type`: 83% nulos, resto constante.
- `holiday_name`: 100% nulos.

El archivo generado es:
`data/processed/station_status_history_2022_modeling_base_clean.csv`

In [29]:
# ============================================================
# 25. Generación de dataset limpio para modelado
# ============================================================

columns_to_drop = [
    "station_number",
    "station_light_status",
    "source_month",
    "date",
    "month",
    "day_of_week",
    "is_non_working_day",
    "is_holiday",
    "holiday_type",
    "holiday_name",
]

clean_output_path = config.processed_dir / "station_status_history_2022_modeling_base_clean.csv"

if processed_modeling_base_output_path.exists():
    df_modeling = pd.read_csv(processed_modeling_base_output_path)
    
    # Verificar que las columnas existen antes de eliminar
    existing_cols_to_drop = [c for c in columns_to_drop if c in df_modeling.columns]
    missing_cols = [c for c in columns_to_drop if c not in df_modeling.columns]
    
    if missing_cols:
        print(f"Columnas no encontradas (ya eliminadas?): {missing_cols}")
    
    df_clean = df_modeling.drop(columns=existing_cols_to_drop)
    
    df_clean.to_csv(clean_output_path, index=False)
    
    print(f"Archivo limpio generado: {clean_output_path.relative_to(config.project_root)}")
    print(f"Columnas originales: {len(df_modeling.columns)}")
    print(f"Columnas eliminadas: {len(existing_cols_to_drop)}")
    print(f"Columnas finales: {len(df_clean.columns)}")
    print(f"Filas: {len(df_clean):,}")
    
    # Mostrar columnas finales
    print("\nColumnas finales:")
    for i, col in enumerate(df_clean.columns, 1):
        print(f"  {i:2d}. {col}")
else:
    print(f"No existe el archivo base: {processed_modeling_base_output_path.relative_to(config.project_root)}")
    print("Ejecuta primero la sección 12 para generar la base de modelado.")

C:\Users\elena\AppData\Local\Temp\ipykernel_4240\2640202825.py:21: DtypeWarning: Columns (0: holiday_type) have mixed types. Specify dtype option on import or set low_memory=False.
  df_modeling = pd.read_csv(processed_modeling_base_output_path)


Archivo limpio generado: data\processed\station_status_history_2022_modeling_base_clean.csv
Columnas originales: 41
Columnas eliminadas: 10
Columnas finales: 31
Filas: 2,306,832

Columnas finales:
   1. snapshot_datetime
   2. station_id_historical
   3. station_name
   4. station_address
   5. station_latitude
   6. station_longitude
   7. station_capacity
   8. num_docks_available
   9. num_bikes_available
  10. reservations_count
  11. is_active
  12. is_not_available
  13. occupation_ratio
  14. snapshot_hour
  15. snapshot_day_of_week
  16. snapshot_month
  17. year
  18. day_of_week_name_es
  19. day_type
  20. is_working_day
  21. is_weekend
  22. weather_temperature_mean_c
  23. weather_temperature_min_c
  24. weather_temperature_max_c
  25. weather_precipitation_mm
  26. weather_humidity_mean_pct
  27. weather_station_count
  28. weather_temperature_available_count
  29. weather_precipitation_available_count
  30. weather_humidity_available_count
  31. weather_has_precipitatio